# **Dự Đoán Chất Lượng Không Khí Cho Ngày Hôm Sau Ở TP.HCM**

Chất lượng không khí tại TP. Hồ Chí Minh ngày càng trở thành mối quan tâm lớn của người dân, đặc biệt với chỉ số AQI trung bình năm 2022-2026 đạt 81.6 (mức Trung bình theo thang US), trong đó nhiều ngày vượt ngưỡng 100 - mức Không tốt cho nhóm nhạy cảm. Khả năng dự báo AQI trước một ngày giúp người dân lên kế hoạch sinh hoạt, hạn chế phơi nhiễm, và hỗ trợ các cơ quan quản lý môi trường.

### **Mục tiêu dự án**
- Thực hiện EDA trên bộ dữ liệu chất lượng không khí TPHCM giai đoạn 2022-2026 và phân tích tính mùa vụ bằng Time Series Decomposition.
- Xây dựng và so sánh nhiều mô hình Machine Learning để dự đoán chất lượng không khí của ngày hôm sau.
- Giải thích mô hình bằng SHAP Values - Trả lời câu hỏi "tại sao model dự đoán vậy".
- Tích hợp API thời gian thực (Open-Meteo) để lấy dữ liệu cuối ngày và tự động đưa ra dự đoán.
- Xây dựng Dashboard Dash/Plotly trực quan với 7 biểu đồ tương tác.
- Trình bày kết quả dưới dạng báo cáo, notebook Jupyter và các file python.

---

### **Phạm vi dự án**
- **Địa điểm:** TP. Hồ Chí Minh (10.82°N, 106.63°E) - dân số ~14 triệu người (2026).
- **Giai đoạn:** 01/08/2022 - 18/02/2026 (1.298 ngày, tần suất ngày).
- **Biến mục tiêu:** US AQI ngày t+1 (ngày hôm sau).

---

### **Các mô hình được triển khai**

| Mô hình | Lý do chọn |
| --- | --- |
| LightGBM | Nhanh, chính xác cao, ít nhạy cảm với outliers, phù hợp Tabular Time Series|
| XGBoost | Phổ biến, nhiều tài liệu, dễ tích hợp SHAP, phù hợp báo cáo học thuật |
| Random Forest | Ổn định, giải thích được qua feature importance |
| Decision Tree | Baseline cho các mô hình cây quyết định khác |
| Ridge Regression | Baseline nhanh để so sánh |
| SARIMA | Mô hình thống kê cổ điển để chứng minh ML vượt trội |

> **Dự đoán mô hình tốt nhất:** LightGBM hoặc XGBoost.

---

### **Cấu trúc notebook**
1. Khám phá dữ liệu (EDA)
2. Tiền xử lý (Preprocessing)
3. Model Regression (Bài toán Hồi quy)
4. Model Classification (Bài toán Phân loại)
5. SHAP Explainability (Giải thích Mô hình Hồi quy)

## **00. Import và cấu hình**

In [ ]:
# Standard Libraries
import warnings
import gdown
import os
import pickle
import joblib
import json
import copy
from google.colab import drive
from datetime import datetime

# Data Manipulation - Math
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import gaussian_kde
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colorbar import ColorbarBase

# Machine Learning - Preprocessing
import shap
import xgboost as xgb
import lightgbm as lgb

from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBClassifier

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

!pip install pmdarima
from pmdarima import auto_arima

from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay
)

In [ ]:
# Cấu hình
warnings.filterwarnings(
    "ignore"
)  # Tắt các cảnh báo không cần thiết cho notebook sạch hơn
pd.set_option(
    "display.float_format", "{:.2f}".format
)  # Hiển thị số thập phân gọn hơn (làm tròn 2 chữ số thập phân)
plt.rcParams.update(
    {
        "figure.dpi": 150,
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.family": "DejaVu Sans",
    }
)  # Cấu hình style mặc định cho toàn bộ biểu đồ matplotlib trong notebook

# AQI Color Palette
# Đây là màu chính thức của US EPA, để đảm bảo nhất quán toàn bộ notebook
AQI_COLORS = {
    "Good": "#00E400",
    "Moderate": "#FFFF00",
    "Unhealthy for Sensitive Groups": "#FF7E00",
    "Unhealthy": "#FF0000",
    "Very Unhealthy": "#8F3F97",
    "Hazardous": "#7E0023",
}

AQI_BINS = [0, 50, 100, 150, 200, 300, 500]
AQI_LABELS = list(AQI_COLORS.keys())
AQI_MAPPING = {label: idx for idx, label in enumerate(AQI_LABELS)}


# Hàm AQI_CATEGORY()
def AQI_CATEGORY(value):
    """
    Phân loại mức AQI theo thang US EPA
    Input : Giá trị AQI (số thực)
    Output: Tên mức (string)
    """
    if value <= 50:
        return "Good"
    elif value <= 100:
        return "Moderate"
    elif value <= 150:
        return "Unhealthy for Sensitive Groups"
    elif value <= 200:
        return "Unhealthy"
    elif value <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"


# Tạo Custom Gradient Colormap từ AQI_COLORS và AQI_BINS
# Chuẩn hóa các mốc AQI vì LinearSegmentedColormap là thang đo 0.0 đến 1.0
MAX_AQI = 500
positions = [
    val / MAX_AQI for val in AQI_BINS
]  # positions sẽ trở thành [0.0, 0.1, 0.2, 0.3, 0.4, 0.6, 1.0]

# Vì AQI_BINS có 7 mốc nhưng AQI_COLORS chỉ có 6 màu
colors = list(AQI_COLORS.values())
gradient_colors = colors + [colors[-1]]  # Nhân đôi màu cuối cùng

# Ghép vị trí và màu sắc lại để tạo dải gradient
color_mapping = list(
    zip(positions, gradient_colors)
)  # Gán các màu tương ứng với từng giá trị
AQI_gradient_cmap = LinearSegmentedColormap.from_list(
    "AQI_gradient", color_mapping
)  # Tạo ra dãy màu liên tục thay vì riêng lẻ

In [ ]:
drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/HCMUS/Nhập Môn Khoa Học Dữ Liệu/Mini Project"

figures_dir = os.path.join(ROOT, "figures")
eda_dir = os.path.join(figures_dir, "eda")
preprocessing_dir = os.path.join(figures_dir, "preprocessing")
models_dir = os.path.join(figures_dir, "models")

os.makedirs(figures_dir, exist_ok=True)
os.makedirs(eda_dir, exist_ok=True)
os.makedirs(preprocessing_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

## **01. Khám Phá Dữ Liệu (EDA)**

#### **Mục tiêu:**
- Hiểu cấu trúc và chất lượng dữ liệu.
- Phát hiện xu hướng, tính mùa vụ và các mẫu đặc trưng.
- Cung cấp insight để định hướng feature engineering ở notebook 02.

#### **Cấu trúc:**
1. Đọc File dữ liệu và nhận xét sơ lược.
2. Thống kê mô tả, đánh giá và kiểm tra dữ liệu.
3. Phân phối US AQI.
4. Chuỗi thời gian.
5. Tính mùa vụ.
6. Time Series Decomposition.
7. So sánh AQI theo năm 2022-2026.
8. Mối tương quan.
9. Bản đồ Folium TPHCM.
10. Tổng kết.

### **1.1. Đọc File dữ liệu và nhận xét sơ lược**

In [ ]:
folder_ID = "1_5C0jQ9fI_ALET0DJPypVJeKS4Cxjtt7"
folder_url = f"https://drive.google.com/drive/folders/{folder_ID}"

gdown.download_folder(folder_url, output="data", quiet=False)

df_city_info = pd.read_csv("data/city_info.csv")
df_data_dictionary = pd.read_csv("data/data_dictionary.csv")
df_air_quality_historical = pd.read_csv("data/air_quality_historical.csv")

In [ ]:
df_city_info

In [ ]:
df_data_dictionary

In [ ]:
df_air_quality_historical

#### **Tổng quan dữ liệu:**
Dữ liệu về chất lượng không khí của thành phố Hồ Chí Minh gồm 1298 dòng (01/08/2022 - 18/02/2026) và 12 cột (các chỉ số đánh giá chất lượng không khí và ngày đánh giá).

### **1.2. Thống kê mô tả, đánh giá và kiểm tra dữ liệu**

In [ ]:
df_air_quality_historical.describe().round(3)

In [ ]:
info_df = pd.DataFrame(
    {
        "Kiểu dữ liệu": df_air_quality_historical.dtypes,
        "Giá trị thiếu": df_air_quality_historical.isnull().sum(),
        "Tỷ lệ thiếu (%)": (
            df_air_quality_historical.isnull().sum()
            / len(df_air_quality_historical)
            * 100
        ).round(3),
    }
)
print(info_df.to_string())

In [ ]:
# Chuyển đổi cột 'date' sang định dạng datetime và trích xuất ngày, tháng, năm
df_air_quality_historical["date"] = pd.to_datetime(df_air_quality_historical["date"])
df_air_quality_historical["day"] = df_air_quality_historical["date"].dt.day
df_air_quality_historical["month"] = df_air_quality_historical["date"].dt.month
df_air_quality_historical["year"] = df_air_quality_historical["date"].dt.year

In [ ]:
# Vẽ histogram với màu theo từng ngưỡng AQI
aqi_data = df_air_quality_historical["us_aqi"].dropna()
fig, ax = plt.subplots(figsize=(12, 6))

# Tính bins thủ công để gán màu từng cột
n_bins = 30
counts, bin_edges = np.histogram(aqi_data, bins=n_bins)

for i in range(len(counts)):
    bin_mid = (bin_edges[i] + bin_edges[i + 1]) / 2
    bin_width = bin_edges[i + 1] - bin_edges[i]
    normalized_val = bin_mid / MAX_AQI
    bar_color = AQI_gradient_cmap(normalized_val)

    # Màu chữ cho viền
    edge_color = "gray"

    ax.bar(
        bin_edges[i],
        counts[i],
        width=bin_width,
        align="edge",
        color=bar_color,
        edgecolor=edge_color,
        linewidth=0.5,
        alpha=0.92,
    )

# Đường KDE
kde_x = np.linspace(aqi_data.min(), aqi_data.max(), 300)
kde = gaussian_kde(aqi_data, bw_method="scott")
# Scale KDE về cùng trục tần suất (Count)
kde_y = kde(kde_x) * len(aqi_data) * (aqi_data.max() - aqi_data.min()) / n_bins
ax.plot(kde_x, kde_y, color="#1a1a2e", linewidth=2.2, label="KDE")

# Vẽ đường phân cách ngưỡng AQI
for threshold, label in zip(AQI_BINS[1:-1], AQI_LABELS[1:]):
    if threshold <= aqi_data.max():
        ax.axvline(threshold, color="#333333", linestyle="--", linewidth=0.8, alpha=0.6)

# Legend màu AQI
patches = [
    mpatches.Patch(facecolor=AQI_COLORS[lbl], edgecolor="#666666", label=lbl)
    for lbl in AQI_LABELS
    if aqi_data.max() >= AQI_BINS[AQI_LABELS.index(lbl)]
]
ax.legend(
    handles=patches,
    title="Mức AQI (US EPA)",
    loc="upper right",
    fontsize=9,
    title_fontsize=10,
    framealpha=0.9,
)

ax.set_title("Phân Phối Biến US AQI", fontsize=14, pad=15, fontweight="bold")
ax.set_xlabel("US AQI", fontsize=11)
ax.set_ylabel("Tần suất", fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
# file_path = os.path.join(eda_dir, 'aqi_dist_hist')
# plt.savefig(file_path, bbox_inches='tight')
# print("Đã lưu biểu đồ thành công")
plt.show()

In [ ]:
aqi_data = (
    df_air_quality_historical["us_aqi"].dropna().sort_values().reset_index(drop=True)
)
total_samples = len(aqi_data)


# Tính toán số lượng và % để đưa vào Chú thích
aqi_categories = aqi_data.apply(AQI_CATEGORY)
aqi_counts = aqi_categories.value_counts().reindex(AQI_LABELS).fillna(0)

legend_labels = []
for label in AQI_LABELS:
    count = aqi_counts[label]
    pct = (count / total_samples) * 100
    legend_labels.append(f"{label} ({pct:.2f}%)")

N = 360
indices = np.linspace(0, total_samples - 1, N).astype(int)
smooth_aqi = aqi_data.iloc[indices].values
slice_colors = [AQI_gradient_cmap(val / MAX_AQI) for val in smooth_aqi]

# Vẽ biểu đồ tròn
fig, ax = plt.subplots(figsize=(10, 6))
wedges, _ = ax.pie(
    [1] * N,
    colors=slice_colors,
    startangle=90,
    counterclock=True,
    wedgeprops={"linewidth": 0, "edgecolor": "none"},
)

# Chuyển đổi toàn bộ các lát cắt thành tên danh mục tương ứng
smooth_categories = [AQI_CATEGORY(val) for val in smooth_aqi]

for i in range(N - 1):
    # Nếu lát cắt hiện tại và lát cắt kế tiếp khác nhóm nhau sẽ vẽ ranh giới
    if smooth_categories[i] != smooth_categories[i + 1]:
        # Tính góc độ của đường ranh giới (mỗi lát tương ứng 1 độ, xuất phát từ góc 90)
        angle_deg = 90 + (i + 1)
        angle_rad = np.radians(angle_deg)

        # Tính tọa độ điểm x, y trên rìa đường tròn (Bán kính R = 1)
        x = np.cos(angle_rad)
        y = np.sin(angle_rad)

        # Vẽ đường phân cách nét liền từ tâm (0,0) ra rìa (x,y)
        ax.plot(
            [0, x],
            [0, y],
            color="#2c3e50",
            linestyle="-",
            linewidth=0.5,
            alpha=0.9,
            zorder=3,
        )

outer_border = plt.Circle(
    (0, 0), 1.0, edgecolor="#2c3e50", fill=False, linewidth=0.5, zorder=3
)
ax.add_artist(outer_border)

legend_patches = [
    mpatches.Patch(
        facecolor=AQI_COLORS[lbl], edgecolor="#666666", label=legend_labels[i]
    )
    for i, lbl in enumerate(AQI_LABELS)
    if aqi_counts[lbl] > 0
]

ax.legend(
    handles=legend_patches,
    title="Mức độ US AQI và Tỷ lệ",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    fontsize=11,
    title_fontsize=11,
    framealpha=0.9,
)
ax.set_title("Phân Phối Mức Độ US AQI", fontsize=14, fontweight="bold", pad=15)

plt.tight_layout()
file_path = os.path.join(eda_dir, "aqi_dist_pie")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

### **1.4. Chuỗi thời gian**

In [ ]:
# Đưa cột 'date' làm index và thiết lập tần suất hàng ngày (D)
df_freq = df_air_quality_historical.set_index("date")["us_aqi"].asfreq("D")

# Xử lý dữ liệu bị thiếu bằng cách điền bù tuyến tính (interpolate) và fill ngược (bfill)
df_freq = df_freq.interpolate(method="linear").bfill()

df_rolling = pd.DataFrame({"Daily AQI": df_freq})
# Tính trung bình động (Rolling Mean) cho 7 ngày và 30 ngày
df_rolling["7-day MA"] = df_freq.rolling(window=7, min_periods=1).mean()
df_rolling["30-day MA"] = df_freq.rolling(window=30, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 7))

# Vẽ đường AQI gốc (Hàng ngày)
ax.plot(
    df_rolling.index,
    df_rolling["Daily AQI"],
    label="AQI Hàng ngày",
    color="#FF9800",
    linewidth=1,
    alpha=0.5,
)

# Vẽ đường Rolling Mean 7 ngày
ax.plot(
    df_rolling.index,
    df_rolling["7-day MA"],
    label="Trung bình động 7 ngày",
    color="#4CAF50",
    linewidth=1.5,
    alpha=0.9,
)
# Vẽ đường Rolling Mean 30 ngày (Xu hướng dài hạn)
ax.plot(
    df_rolling.index,
    df_rolling["30-day MA"],
    label="Trung bình động 30 ngày",
    color="#F44336",
    linewidth=2.5,
)

# Định dạng các thành phần của biểu đồ
ax.set_title(
    "Biến Động US AQI Và Các Đường Trung Bình Động (Rolling Mean)",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
ax.set_xlabel("Thời gian", fontsize=11)
ax.set_ylabel("Chỉ số US AQI", fontsize=11)

ax.legend(
    loc="lower right", frameon=True, fontsize=11, facecolor="white", framealpha=0.9
)

plt.tight_layout()
file_path = os.path.join(eda_dir, "aqi_rolling_mean")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

### **1.5. Tính mùa vụ**

#### **Boxplot**

In [ ]:
sns.set_theme(style="whitegrid")
# Định nghĩa tên các tháng để hiển thị trực quan trên trục x
month_labels = [
    "Tháng 1",
    "Tháng 2",
    "Tháng 3",
    "Tháng 4",
    "Tháng 5",
    "Tháng 6",
    "Tháng 7",
    "Tháng 8",
    "Tháng 9",
    "Tháng 10",
    "Tháng 11",
    "Tháng 12",
]

# Thiết lập giao diện
fig, ax = plt.subplots(figsize=(10, 6))

# Vẽ Boxplot cho trung bình US AQI của 5 năm
sns.boxplot(
    x="month",
    y="us_aqi",
    data=df_air_quality_historical,
    color="#e0e0e0",
    ax=ax,
    width=0.6,
)

# Vẽ đường xu hướng cho US AQI qua từng năm
df_monthly = (
    df_air_quality_historical.groupby(["year", "month"])["us_aqi"].mean().reset_index()
)
sns.lineplot(
    x=df_monthly["month"] - 1,
    y="us_aqi",
    hue="year",
    data=df_monthly,
    marker="o",
    linewidth=2.5,
    palette="tab10",
    ax=ax,
)

ax.set_title(
    "Phân Phối Chỉ Số US AQI Theo Từng Tháng Trong Năm",
    fontsize=14,
    pad=15,
    fontweight="bold",
)
ax.set_xlabel("Tháng", fontsize=12)
ax.set_ylabel("US AQI", fontsize=12)

ax.set_xticks(range(12))
ax.set_xticklabels(month_labels, rotation=45)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.4), ncol=5, framealpha=0.0)


plt.tight_layout()
file_path = os.path.join(eda_dir, "aqi_monthly_dist")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Heatmap**

In [ ]:
# Lệnh unstack() giúp xoay dữ liệu thành dạng ma trận: hàng là Tháng, cột là Ngày
df_daily = (
    df_air_quality_historical.groupby(["month", "day"])["us_aqi"].mean().unstack()
)
fig, ax = plt.subplots(figsize=(15, 8))

# Vẽ heatmap với dải màu thể hiện mức độ ô nhiễm tăng dần
sns.heatmap(
    df_daily,
    cmap=AQI_gradient_cmap,
    vmin=0,
    vmax=500,
    ax=ax,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Chỉ số US AQI (Chuẩn EPA)"},
)
ax.grid(False)

ax.set_title(
    "Mức Độ Ô Nhiễm (US AQI) Theo Ngày và Tháng", fontsize=14, pad=15, fontweight="bold"
)
ax.set_xlabel("Ngày", fontsize=12)
ax.set_ylabel("Tháng", fontsize=12)

ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
file_path = os.path.join(eda_dir, "aqi_daily_monthly_heatmap")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

### **1.6. Time Series Decomposition**

**Định nghĩa Time Series Decomposition (Phân rã chuỗi thời gian)**: Là một kỹ thuật được sử dụng để phân tách một chuỗi thời gian thành ba thành phần chính - xu hướng (trend), tính thời vụ (seasonality) và residuals (nhiễu hay phần dư). Quá trình này giúp hiểu rõ các quy luật tiềm ẩn trong dữ liệu và đóng vai trò cực kỳ quan trọng đối với các tác vụ như dự báo và phát hiện điểm bất thường.

In [ ]:
# Biến đổi date làm index và khai báo tuần suất là hàng ngày (D)
df_freq = df_air_quality_historical.set_index("date")["us_aqi"].asfreq("D")

# Xử lý dữ liệu bị thiếu bằng cách điền bù
df_freq = df_freq.interpolate(method="linear").bfill()

# Thực hiện Decomposition
result = seasonal_decompose(
    df_freq,
    model="additive",  # AQI = Trend + Seasonal + Residual
    period=365,  # Chu kỳ 1 năm
)
# model = 'multiplicative': AQI = Trend × Seasonal × Residual

# Vẽ 4 Subplot cho: Dữ liệu gốc (Observed), Trend, Seasonal và Residual
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle(
    "Time Series Decomposition - US AQI TP.HCM", fontsize=14, fontweight="bold"
)

# Observed
result.observed.plot(ax=axes[0], color="#2196F3", linewidth=0.8)
axes[0].set_ylabel("Observed")
axes[0].set_title("Chuỗi gốc")

# Trend
result.trend.plot(ax=axes[1], color="#FF5722", linewidth=1.8)
axes[1].set_ylabel("Trend")
axes[1].set_title("Xu hướng (Dài hạn)")

# Seasonal
result.seasonal.plot(ax=axes[2], color="#4CAF50", linewidth=0.8)
axes[2].set_ylabel("Seasonal")
axes[2].set_title("Mùa vụ (Chu kỳ 365 ngày)")

# Residual = Observed - (Trend + Seasonal)
# Nếu thành phần Trend và Seasonal giải thích được 100% sự biến động của không khí, thì Residual sẽ bằng đúng 0.
result.resid.plot(ax=axes[3], color="#9C27B0", linewidth=0.8)
axes[3].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[3].set_ylabel("Residual")
axes[3].set_xlabel("Ngày")
axes[3].set_title("Phần dư (Nhiễu ngẫu nhiên)")

plt.tight_layout()
file_path = os.path.join(eda_dir, "aqi_time_series_decomposition")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

### **1.7. So sánh AQI theo năm 2022-2026**

In [ ]:
# Tạo Figure và Axes
fig, ax = plt.subplots(figsize=(12, 6))

# Nhóm dữ liệu theo năm và tháng, tính trung bình AQI
df_monthly = (
    df_air_quality_historical.groupby(["year", "month"])["us_aqi"].mean().reset_index()
)

sns.lineplot(
    data=df_monthly,
    x="month",
    y="us_aqi",
    hue="year",
    palette="tab10",
    marker="o",
    markersize=8,
    linewidth=2.5,
    ax=ax,
)

ax.set_title(
    "So Sánh Xu Hướng US AQI Các Năm (2022-2026)",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
ax.set_xlabel("Tháng", fontsize=11)
ax.set_ylabel("US AQI", fontsize=11)

month_labels = [
    "Tháng 1",
    "Tháng 2",
    "Tháng 3",
    "Tháng 4",
    "Tháng 5",
    "Tháng 6",
    "Tháng 7",
    "Tháng 8",
    "Tháng 9",
    "Tháng 10",
    "Tháng 11",
    "Tháng 12",
]
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels, rotation=0)
ax.legend(
    title="Năm", loc="upper right", frameon=True, facecolor="white", framealpha=0.9
)

plt.tight_layout()
file_path = os.path.join(eda_dir, "aqi_yearly_comparison")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

### **1.8. Mối tương quan**

In [ ]:
# Tạo vùng vẽ với kích thước lớn để chứa đủ các biến
fig, ax = plt.subplots(figsize=(14, 10))

# 1. Lọc ra các cột chứa dữ liệu dạng số (loại bỏ cột ngày tháng dạng string/datetime)
df_numeric = df_air_quality_historical.select_dtypes(include=[np.number])

# Vì trong df_air_quality_historical đã tạo các cột year, month, day, nên loại bỏ chúng ra khỏi ma trận
# vì tương quan tuyến tính của thời gian thường không thể hiện rõ qua Pearson
cols_to_drop = ["year", "month", "day"]
df_numeric = df_numeric.drop(
    columns=[col for col in cols_to_drop if col in df_numeric.columns], errors="ignore"
)

# 2. Tính toán ma trận tương quan Pearson
corr_matrix = df_numeric.corr()


# 3. Vẽ Heatmap
# Sử dụng dải màu 'coolwarm' (Xanh - Đỏ): Xanh là tương quan âm, Đỏ là tương quan dương
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    vmax=1.0,
    vmin=-1.0,
    center=0,
    annot=True,  # Hiển thị con số bên trong ô
    fmt=".2f",  # Làm tròn 2 chữ số thập phân
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.7, "label": "Hệ số tương quan (Pearson)"},
    ax=ax,
)

ax.grid(False)
ax.set_title(
    "Ma Trận Tương Quan Giữa Các Biến Số Khí Tượng Và Ô Nhiễm",
    fontsize=14,
    fontweight="bold",
    pad=15,
)

# Xoay nhãn trục X và Y để chữ không bị đè lên nhau
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
file_path = os.path.join(eda_dir, "meteo_aqi_corr_matrix")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

### **1.9. Bản đồ Folium TPHCM**

In [ ]:
# 1. Dữ liệu tọa độ các trạm quan trắc thực tế tại TP.HCM
data_folium = {
    "Folium": [
        "Lãnh sự quán Mỹ (Quận 1)",
        "RMIT University (Quận 7)",
        "Thảo Điền (Quận 2)",
        "Đại học Bách Khoa (Quận 10)",
        "Khu công nghệ cao (Thủ Đức)",
    ],
    "Lat": [10.7828, 10.7293, 10.8037, 10.7734, 10.8521],
    "Lon": [106.7003, 106.6943, 106.7371, 106.6606, 106.7974],
}
df_folium = pd.DataFrame(data_folium)

# 2. Định nghĩa vùng giới hạn cho TP.HCM
# [[Vĩ độ Nam, Kinh độ Tây], [Vĩ độ Bắc, Kinh độ Đông]]
HCM_bounds = [[10.3500, 106.3500], [11.1500, 107.0500]]

# 3. Khởi tạo bản đồ
HCM_map = folium.Map(
    location=[10.82302, 106.62965],  # Trung tâm TP.HCM
    zoom_start=12,
    tiles="CartoDB positron",
    min_zoom=11,
    max_zoom=16,
    max_bounds=True,
    bounds=HCM_bounds,
)

# 4. Đánh dấu các trạm
for idx, row in df_folium.iterrows():
    folium.CircleMarker(
        location=[row["Lat"], row["Lon"]],
        radius=8,
        popup=folium.Popup(f"<b>Trạm quan trắc:</b> {row['Folium']}", max_width=250),
        tooltip="Nhấn để xem chi tiết",
        color="#3186cc",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.8,
    ).add_to(HCM_map)

file_path = os.path.join(eda_dir, "monitoring_stations_map.html")
HCM_map.save(file_path)
print("Đã lưu bản đồ tương tác thành công")

# Hiển thị bản đồ
HCM_map

### **1.10. Tổng kết**

#### **Các phát hiện chính:**
- **Chất lượng không khí TPHCM ở mức trung bình** - AQI trung bình 81.6,
   phần lớn thời gian (76.3%) ở mức Moderate. Gần 1/5 số ngày (18.6%)
   ảnh hưởng đến nhóm nhạy cảm.

- **Tính mùa vụ rõ ràng** - Mùa khô (T11-T4) AQI cao hơn mùa mưa (T5-T10)
   do thiếu mưa rửa trôi bụi và hiện tượng nghịch nhiệt. Biên độ dao động
   khoảng ±30-45 đơn vị AQI theo mùa.

- **Xu hướng đang xấu dần** - Từ đầu 2024 đến giữa 2025, đường Trend
   tăng liên tục từ ~70 lên >90. Năm 2025 là năm ô nhiễm nhất trong
   toàn bộ chuỗi dữ liệu.

- **PM2.5 là yếu tố chi phối AQI** - Tương quan r≈0.92,
   cao hơn hẳn các chất khác. Mọi nỗ lực cải thiện không khí cần
   ưu tiên kiểm soát PM2.5.

- **Nhiễu ngẫu nhiên lớn** - Residual có spike đến ±40 đơn vị,
   phản ánh các sự kiện cục bộ khó dự đoán.

#### **Định hướng Feature Engineering:**
- Mã hóa tính mùa vụ (Tuần hoàn sin/cos).
- Tạo `is_dry_season` (T11-T4 = 1) vì biên độ mùa vụ lớn.
- Lag features `pm2_5_lag1`, `us_aqi_lag1` vì PM2.5 tương quan cao nhất.
- Rolling mean 7 ngày (hoặc 30 ngày) để nắm bắt xu hướng dài hạn.

## **02. Tiền Xử Lý (Preprocessing)**

#### **Mục tiêu:**
- Làm sạch dữ liệu: Xử lý missing values và outliers.
- Tạo features từ chuỗi thời gian: Lag, rolling, đặc trưng thời gian, sin/cos.
- Tạo biến mục tiêu cho cả Regression và Classification.
- Chia tập Train/Test theo thời gian (không shuffle).
- Chuẩn hóa dữ liệu bằng RobustScaler.
- Lưu các file cần thiết cho huấn luyện mô hình: `train.csv`, `test.csv`, `scaler.pkl`, `feature_cols.pkl`.

#### **Cấu trúc:**
1. Xử lý dữ liệu và outliers
2. Lag Features - Rolling Features
3. Đặc trưng thời gian
4. Biến mục tiêu
5. Chia tập Train và Test
6. Xử lý outliers trên tập Train bằng Winsorize
7. Chuẩn hóa dữ liệu
8. Lưu file
9. Tổng kết


In [ ]:
df = pd.read_csv("data/air_quality_historical.csv", parse_dates=["date"])
df_air_quality_historical = df.set_index("date").asfreq("D")

df_air_quality_historical.head()

### **2.1. Xử lý dữ liệu và outliers**

#### **Bỏ eurpean_aqi vì đã sử dụng us_aqi**

Dự án dùng `us_aqi` làm biến mục tiêu theo chuẩn US EPA - phổ biến hơn tại Việt Nam và dễ so sánh với IQAir, AQICN. Cột `european_aqi` không đưa vào model.

In [ ]:
df_air_quality_historical.drop(columns=["european_aqi"], inplace=True)
df_air_quality_historical.head()

#### **Xử lý missing values**

In [ ]:
df_air_quality_historical.isnull().sum()

In [ ]:
df_air_quality_historical[df_air_quality_historical.isnull().any(axis=1)]

In [ ]:
# Điền bù vẫn nên được ưu tiên hơn so với loại bỏ với dữ liệu mang tính thời gian cho dù nó có là những dữ liệu đầu tiên
df_air_quality_historical.interpolate(
    method="time", limit_direction="both", inplace=True
)
df_air_quality_historical.isnull().sum()

In [ ]:
df_air_quality_historical.head()

#### **Kiểm tra outliers**

In [ ]:
cols = df_air_quality_historical.select_dtypes(include="number").columns.tolist()
n = len(cols)
nrows = (n + 4) // 5

plt.figure(figsize=(16, 4 * nrows))
for i, col in enumerate(cols, 1):
    plt.subplot(nrows, 5, i)
    sns.boxplot(y=df_air_quality_historical[col], color="#AED6F1")
    plt.title(col, fontsize=9)
    plt.ylabel("")

plt.suptitle("Boxplot Các Chỉ Số - Kiểm Tra Outliers Trước Xử Lý", fontweight="bold")
plt.tight_layout()
file_path = os.path.join(preprocessing_dir, "boxplot_check_outliers")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

In [ ]:
numeric_cols = df_air_quality_historical.select_dtypes(
    include=["float64", "int64"]
).columns
outlier_data = []
N = len(df_air_quality_historical)

for col in numeric_cols:
    Q1 = df_air_quality_historical[col].quantile(0.25)
    Q3 = df_air_quality_historical[col].quantile(0.75)
    IQR = Q3 - Q1
    # IQR×1.5: Ngưỡng chuẩn thống kê - dùng để PHÁT HIỆN và báo cáo tỷ lệ
    n_15 = (
        (df_air_quality_historical[col] < Q1 - 1.5 * IQR)
        | (df_air_quality_historical[col] > Q3 + 1.5 * IQR)
    ).sum()
    # IQR×3.0: Ngưỡng conservative - dùng để XỬ LÝ, tránh mất nhiều dữ liệu
    n_30 = (
        (df_air_quality_historical[col] < Q1 - 3.0 * IQR)
        | (df_air_quality_historical[col] > Q3 + 3.0 * IQR)
    ).sum()

    outlier_data.append(
        {
            "Chỉ số": col,
            "Q1": round(Q1, 2),
            "Q3": round(Q3, 2),
            "Outliers (IQR×1.5)": f"{n_15} ({n_15 / N * 100:.1f}%)",
            "Outliers (IQR×3.0)": f"{n_30} ({n_30 / N * 100:.1f}%)",
        }
    )

df_outlier_summary = pd.DataFrame(outlier_data).sort_values(
    "Outliers (IQR×1.5)", ascending=False
)
display(df_outlier_summary.style.hide(axis="index"))

> **Quyết định:** Winsorize dùng ngưỡng `IQR*3.0` vì các ngày AQI cao (130-158) là sự kiện ô nhiễm thật, không phải lỗi đo lường. Xử lý sẽ được thực hiện sau khi chia Train/Test - tránh **data leakage** từ tập Test vào quá trình tính ngưỡng.

### **2.2. Lag Features - Rolling Features**

Model cần dự đoán AQI ngày mai. Nhưng nếu chỉ đưa vào các chỉ số đo ngày hôm nay (pm2_5, pm10, ozone...) thì model không biết xu hướng đang đi lên hay đi xuống nên Lag và Rolling là cách đưa lịch sử vào làm feature để model có ngữ cảnh.

In [ ]:
df_air_quality_historical.head()

#### **Tạo Lag Features: us_aqi tại t-1, t-2, t-3, t-7, t-14**

Trả lời cho câu hỏi "Hôm qua là bao nhiêu?"
- t-1: AQI của ngày hôm qua
- t-7: AQI của tuần trước cùng ngày

In [ ]:
lags = [1, 2, 3, 7, 14]

for lag in lags:
    df_air_quality_historical[f"t-{lag}"] = df_air_quality_historical["us_aqi"].shift(
        lag
    )

df_air_quality_historical[[f"t-{l}" for l in lags]].head(5)

**Dự đoán:** `t-1` sẽ có tương quan cao nhất với TARGET vì điều kiện khí quyển thay đổi chậm trong 24h. `t-7` nắm bắt pattern tuần. NaN ở đầu chuỗi (do shift) sẽ được xử lý khi tạo TARGET.

#### **Tạo Rolling mean/std/max/min: Window = 3, 7, 14, 30 ngày**

Trả lời câu hỏi "Xu hướng gần đây thế nào?"
- rolling_std cho model biết mức độ tin cậy của dự đoán - std thấp thì chuỗi đang ổn định, std cao thì đang bất thường

In [ ]:
windows = [3, 7, 14, 30]  # rolling_1 và rolling_2 vô nghĩa
functions = ["mean", "std", "max", "min"]

for w in windows:
    shifted = df_air_quality_historical["us_aqi"].shift(
        1
    )  # shift(1) trước để tránh data leakage
    # Nếu không shift: rolling(7).mean() tại ngày T sẽ bao gồm giá trị ngày T
    rolling_obj = shifted.rolling(window=w)
    for func in functions:
        df_air_quality_historical[f"rolling_{func}_{w}"] = getattr(rolling_obj, func)()

**Dự đoán:** `rolling_mean_7` tóm tắt xu hướng tuần gần nhất - thường nằm trong top 3 feature quan trọng theo SHAP. `rolling_std_7` đo độ biến động: std cao → ngày hôm nay khó dự đoán hơn. `shift(1)` đảm bảo không có data leakage.

### **2.3. Đặc trưng thời gian**

#### **Tạo sai phân: diff(1) và diff(7) cho AQI**

Thuật ngữ sai phân (differencing) được sử dụng để tính khoảng cách giữa giá trị hiện tại và một giá trị trong quá khứ. Mục đích chính là để làm cho chuỗi dữ liệu trở nên "dừng" (stationary) - tức là loại bỏ các xu hướng (trend) hoặc tính mùa vụ (seasonality) để phục vụ cho việc dự báo.

- **diff(1):** Nó cho biết tốc độ thay đổi của chất lượng không khí từ ngày này sang ngày khác. Nếu diff(1) dương, không khí đang ô nhiễm hơn so với hôm qua. Nếu âm, không khí đang trong lành hơn.

- **diff(7):** Nó so sánh mức độ ô nhiễm của thứ Hai tuần này với thứ Hai tuần trước, thứ Ba tuần này với thứ Ba tuần trước, v.v.

In [ ]:
# 1. Tạo sai phân 1 ngày cho chỉ số us_aqi
df_air_quality_historical["diff_1"] = df_air_quality_historical["us_aqi"].diff(
    periods=1
)

# 2. Tạo sai phân 7 ngày cho chỉ số us_aqi
df_air_quality_historical["diff_7"] = df_air_quality_historical["us_aqi"].diff(
    periods=7
)

#### **Tạo đặc trưng thời gian**

In [ ]:
month = df_air_quality_historical.index.month
weekday = df_air_quality_historical.index.dayofweek
day_of_year = df_air_quality_historical.index.dayofyear

In [ ]:
# Mã hóa tuần hoàn sin/cos (Cyclical Encoding) là cách biến các biến thời gian có tính chu kỳ như:
## Tháng (1 → 12)
## Ngày trong tuần (Thứ 2 → Chủ nhật)
## Ngày trong năm (1 → 365)
# Thành 2 đặc trưng mới bằng hàm sin và cos để mô hình hiểu được tính tuần hoàn

# 1. Month
df_air_quality_historical["month_sin"] = np.sin(2 * np.pi * month / 12)
df_air_quality_historical["month_cos"] = np.cos(2 * np.pi * month / 12)

# 2. Weekday
df_air_quality_historical["weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
df_air_quality_historical["weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

# 3. Day of year
df_air_quality_historical["day_of_year_sin"] = np.sin(2 * np.pi * day_of_year / 365)
df_air_quality_historical["day_of_year_cos"] = np.cos(2 * np.pi * day_of_year / 365)

#### **Tạo biến nhị phân: is_weekend, is_dry_season**

In [ ]:
# 1. Tạo biến is_weekend (Cuối tuần)
df_air_quality_historical["is_weekend"] = (weekday >= 5).astype(int)  # 5: T7 - 6: CN

# 2. Tạo biến is_dry_season (Mùa khô)
# Mùa khô ở TP.HCM thường kéo dài từ tháng 12 đến tháng 4 năm sau
dry_month = [12, 1, 2, 3, 4]
df_air_quality_historical["is_dry_season"] = month.isin(dry_month).astype(int)

### **2.4. Biến mục tiêu**

In [ ]:
# 1. Regression Target: Giá trị AQI (số thực) ngày mai
df_air_quality_historical["target_reg_tomorrow"] = df_air_quality_historical[
    "us_aqi"
].shift(-1)

# 2. Classification Target: Mức độ AQI ngày mai (0=Good, 1=Moderate, ...)
df_air_quality_historical["target_class_tomorrow"] = (
    pd.cut(
        df_air_quality_historical["target_reg_tomorrow"],
        bins=AQI_BINS,
        labels=AQI_LABELS,
        right=True,
    )
    .map(AQI_MAPPING)
    .astype("Int64")  # Dịch nhãn chữ sang số nguyên (0, 1, 2...) bằng AQI_MAPPING
)

# Bỏ tất cả hàng có NaN - do lag/rolling tạo ra ở đầu chuỗi
df_air_quality_historical.dropna(inplace=True)
# Dòng cuối cùng chắc chắn sẽ bị NaN do shift
df_air_quality_historical.dropna(subset=["target_reg_tomorrow"], inplace=True)

print(f"Shape sau khi tạo target: {df_air_quality_historical.shape}")
print(f"\nPhân bố target_class_tomorrow:")
counts = df_air_quality_historical["target_class_tomorrow"].value_counts().sort_index()
for idx, cnt in counts.items():
    pct = cnt / len(df_air_quality_historical) * 100
    print(f"  {int(idx)} - {AQI_LABELS[int(idx)]:<35}: {cnt:4d} ngày ({pct:.1f}%)")

# In ra xem thử kết quả
pd.set_option("display.max_columns", None)
df_air_quality_historical[
    ["us_aqi", "target_reg_tomorrow", "target_class_tomorrow"]
].head(5)


In [ ]:
df_air_quality_historical.describe()

### **2.5. Chia tập Train và Test**

In [ ]:
# 1. Tập X
# Bỏ khỏi X:
## - 'us_aqi'                : TARGET gốc, model không được nhìn thấy khi dự đoán
## - 'target_reg_tomorrow'   : Là y regression
## - 'target_class_tomorrow' : Là y classification
## - 'date'                  : không cần sau khi đã có sin/cos
X = df_air_quality_historical.drop(
    columns=["us_aqi", "target_reg_tomorrow", "target_class_tomorrow", "date"],
    errors="ignore",
)

# 2. Tập y (Gom cả 2 biến mục tiêu của ngày hôm sau)
y = df_air_quality_historical[["target_reg_tomorrow", "target_class_tomorrow"]]

# 3. Chia train test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False,  # Vì là dữ liệu có tính thời gian nên shuffle = False
)

print(
    f"Train: {len(X_train):,} mẫu | {X_train.index.min().date()} → {X_train.index.max().date()}"
)
print(
    f"Test : {len(X_test):,} mẫu   | {X_test.index.min().date()} → {X_test.index.max().date()}"
)
print(f"Số features tập X: {X_train.shape[1]}")

### **2.6. Xử lý outliers trên tập Train bằng Winsorize**

In [ ]:
# Winsorize
air_quality_cols = [
    "pm10",
    "pm2_5",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "us_aqi",
]
cols_to_winsorize = [col for col in air_quality_cols if col in X_train.columns]
train_fences = {}

# 1. Tính ngưỡng trên Train - tránh Data Leakage từ Test
for col in cols_to_winsorize:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    train_fences[col] = {
        "lower": max(Q1 - 3.0 * IQR, 0),  # AQI không âm
        "upper": Q3 + 3.0 * IQR,
    }

for col in cols_to_winsorize:
    lo = train_fences[col]["lower"]
    up = train_fences[col]["upper"]
    X_train[col] = X_train[col].clip(lo, up)
    X_test[col] = X_test[col].clip(lo, up)

#### **Lý do không xử lý outliers ở các cột còn lại:**
- `date` - là cột thời gian, không phải số. Không có khái niệm outlier cho ngày tháng và không còn dùng cột date nữa vì đã có sin/cos.
- `aerosol_optical_depth` - đây là chỉ số quang học của khí quyển, giá trị tự nhiên dao động rất rộng và bất thường là bình thường - xử lý outlier sẽ làm phức tạp thêm mà không giúp ích cho model.
- `dust` - tương tự, nồng độ bụi sa mạc có thể tăng đột biến khi có gió mùa mang bụi từ xa đến. Spike cao là sự kiện thật, không phải lỗi đo lường.
- `uv_index` - chỉ số tia UV không đưa vào model dự đoán AQI. Có ít ý nghĩa với bài toán ô nhiễm không khí.

In [ ]:
print(
    f"{'Chỉ số':<25} {'Lower':>10} {'Upper':>10} {'Clip Train':>12} {'Clip Test':>11}"
)
print("-" * 72)
for col in cols_to_winsorize:
    lo = train_fences[col]["lower"]
    up = train_fences[col]["upper"]
    n_train = (X_train[col] == lo).sum() + (X_train[col] == up).sum()
    n_test = (X_test[col] == lo).sum() + (X_test[col] == up).sum()
    print(f"{col:<25} {lo:>10.2f} {up:>10.2f} {n_train:>12} {n_test:>11}")

### **2.7. Chuẩn hóa dữ liệu**


In [ ]:
# 1. Xác định lại các cột cần chuẩn hóa (bao gồm cả Lag và Rolling)
# Không scale các biến phân loại/nhị phân (sin, cos, is_weekend, is_dry_season)
exclude_from_scale = [
    "month_sin",
    "month_cos",
    "weekday_sin",
    "weekday_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    "is_weekend",
    "is_dry_season",
]
all_numeric = X_train.select_dtypes(include=["float64", "int64"]).columns.tolist()
cols_to_scale = [c for c in all_numeric if c not in exclude_from_scale]

# 2. Fit trên Train → transform cả hai - KHÔNG fit lại trên Test
scaler = RobustScaler()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print(f"Số cột được scale  : {len(cols_to_scale)}")
print(f"Số cột không scale : {len(exclude_from_scale)} (sin/cos, nhị phân)")
print(f"Tổng features      : {X_train.shape[1]}")
print(f"\nKiểm tra sau scale (mean và median ≈ 0):")
print(X_train[cols_to_scale[:6]].agg(["mean", "median"]).round(3))
display(X_train.head(3))

#### **Kiểm tra tương quan (Correlation Analysis)**
Việc tạo nhiều đặc trưng Lag và Rolling có thể dẫn đến hiện tượng đa cộng tuyến. Cần kiểm tra mối tương quan giữa các đặc trưng và biến mục tiêu `target_reg_tomorrow`.

In [ ]:
# Ghép tạm thời X_train và y_train để xem tương quan
train_full = pd.concat([X_train, y_train["target_reg_tomorrow"]], axis=1)
corr_matrix = train_full.corr()

# Lấy top 15 đặc trưng tương quan nhất với target
top_features = (
    corr_matrix["target_reg_tomorrow"]
    .drop("target_reg_tomorrow", errors="ignore")
    .abs()
    .sort_values(ascending=False)
    .head(15)
)

top_idx = top_features.index.tolist() + ["target_reg_tomorrow"]

plt.figure(figsize=(12, 10))
sns.heatmap(
    train_full[top_idx].corr(),
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)
plt.grid(False)

plt.title("Ma Trận Tương Quan - Top 15 Features với TARGET", fontweight="bold")
plt.show()


print("Top 15 features tương quan nhất với target_reg_tomorrow:")
for feat, val in top_features.items():
    print(f"  {feat:<35} r={val:.3f}")

### **2.8. Lưu file**

In [ ]:
outputs_dir = os.path.join(ROOT, "outputs")
models_dir = os.path.join(ROOT, "models")
os.makedirs(outputs_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

# Lưu Train - Test
save_path_X_train = os.path.join(outputs_dir, "X_train.csv")
save_path_X_test = os.path.join(outputs_dir, "X_test.csv")
save_path_y_train = os.path.join(outputs_dir, "y_train.csv")
save_path_y_test = os.path.join(outputs_dir, "y_test.csv")

X_train.to_csv(save_path_X_train)
X_test.to_csv(save_path_X_test)
y_train.to_csv(save_path_y_train)
y_test.to_csv(save_path_y_test)

print("Đã lưu X_train, X_test, y_train, y_test")

# data_processed.csv - dùng cho Dashboard và predict.py
save_path = os.path.join(outputs_dir, "data_processed.csv")
df_air_quality_historical.to_csv(save_path)
print("Đã lưu data_processed.csv")

# scaler.pkl - bắt buộc dùng đúng scaler đã fit khi predict
save_path = os.path.join(models_dir, "scaler.pkl")
with open(save_path, "wb") as f:
    pickle.dump(scaler, f)
print("Đã lưu scaler.pkl")

# feature_cols.pkl - danh sách features đúng thứ tự, đảm bảo khi dùng predict.py sẽ đúng thứ tự các features
save_path = os.path.join(models_dir, "feature_cols.pkl")
with open(save_path, "wb") as f:
    pickle.dump(X_train.columns.tolist(), f)
print("feature_cols.pkl")

print(f"\nTổng kết:")
print(f"   X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"   Features: {X_train.columns.tolist()}")

### **2.9. Tổng kết**

Dữ liệu đã sẵn sàng cho bước xây dựng mô hình. Tóm tắt các bước đã thực hiện:

| Bước | Kỹ thuật | Kết quả |
| --- | --- | --- |
| Missing values | `interpolate(method='time', limit_direction='both', inplace=True)` | Điền bù 4 dòng đầu tiên |
| Lag features | `shift(1/2/3/7/14)` | 5 features |
| Rolling features | `shift(1)` + `rolling(3/7/14/30)` | 16 features - không data leakage |
| Sai phân | `diff(1)`, `diff(7)` | 2 features |
| Thời gian | sin/cos encoding | 6 features |
| Nhị phân | `is_weekend`, `is_dry_season` | 2 features |
| TARGET | `shift(-1)` + `pd.cut` | 2 biến mục tiêu |
| Train/Test | 80/20 không shuffle | Train = 1.013, Test = 254 |
| Winsorize | IQR x 3.0 fit trên Train | Không data leakage |
| Chuẩn hóa | `RobustScaler` | Mean ≈ 0, Median ≈ 0 |

#### **Huấn luyện mô hình:** Load `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` từ `outputs/`

Target Regression: `target_reg_tomorrow` | Target Classification: `target_class_tomorrow`

#### **Cái nhìn tổng quan về dữ liệu trước khi huấn luyện mô hình:**

- Dữ liệu TPHCM có tính ngẫu nhiên cao - AQI ngày hôm nay và ngày mai không tương quan chặt chẽ như các thành phố khác vì:

```text
Đặc điểm:
├── Mưa nhiệt đới đột ngột: AQI có thể giảm 40-50 điểm chỉ sau 1 cơn mưa
├── Gió thay đổi nhanh: Phân tán bụi không đều
├── Không có dữ liệu thời tiết: Nhiệt độ, độ ẩm, tốc độ gió
│   → thiếu các yếu tố quan trọng nhất để dự đoán AQI
└── Chỉ có 5 trạm quan trắc → không đại diện toàn thành phố (những chưa khẳng định được là dữ liệu lấy từ bao nhiêu trạm quan trắc)
```

> **Kết luận:** Dự đoán rằng các kết quả từ việc huấn luyện mô hình sẽ không như mong đợi.

## **03. Model Regression**

#### **Mục tiêu:**

- Xây dựng và huấn luyện các mô hình Machine Learning để dự đoán giá trị US AQI ngày hôm sau (bài toán Regression).
- So sánh hiệu suất giữa các mô hình: Ridge Regression, Decision Tree, Random Forest, XGBoost, LightGBM, và SARIMA (mô hình thống kê cổ điển để đối chứng).
- Kiểm tra độ ổn định của từng mô hình bằng Cross-Validation theo thời gian (TimeSeriesSplit), không chỉ dựa vào kết quả Test set đơn.
- Phát hiện và xử lý hiện tượng overfitting (so sánh R² Train vs Test).
- Phân tích Feature Importance để hiểu mô hình dựa vào yếu tố nào để dự đoán.
- Chứng minh giá trị của Feature Engineering bằng cách so sánh ML với SARIMA.
- Lựa chọn Best Model dựa trên cả độ chính xác (R², RMSE, MAE, MAPE) và độ ổn định (CV Mean ± Std).
- Lưu Best Model (`best_regressor.pkl`) và metadata (`metadata.json`) để sử dụng ở SHAP và trong Dashboard.

#### **Cấu trúc:**
1. Ridge Regression (Baseline)
2. Decision Tree
3. Random Forest
4. LightGBM - XGBoost
5. SARIMA
6. So sánh tổng hợp
7. Đánh giá Best Model
8. Phân tích SARIMA với Machine Learning
9. Tổng kết

In [ ]:
folder_ID = "1b8LGMtOwMrj6FDMODA8-rJNq0cKXlgBA"
folder_url = f"https://drive.google.com/drive/folders/{folder_ID}"

gdown.download_folder(folder_url, output="data", quiet=False)

X_train = pd.read_csv("data/X_train.csv", index_col="date", parse_dates=True)
X_test = pd.read_csv("data/X_test.csv", index_col="date", parse_dates=True)
y_train = pd.read_csv("data/y_train.csv", index_col="date", parse_dates=True)
y_test = pd.read_csv("data/y_test.csv", index_col="date", parse_dates=True)
df_processed = pd.read_csv(
    "data/data_processed.csv", index_col="date", parse_dates=True
)

y_train_reg = y_train["target_reg_tomorrow"]
y_test_reg = y_test["target_reg_tomorrow"]

In [ ]:
# Hàm đánh giá
def evaluate(name, y_true, y_pred):
    y_pred = np.clip(y_pred, 0, 500)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"{name} - Kết quả trên tập Test:")
    print(f"   MAE  = {mae:.2f}    (Sai số tuyệt đối trung bình)")
    print(f"   RMSE = {rmse:.2f}   (Căn bậc hai sai số bình phương trung bình)")
    print(f"   MSE  = {mse:.2f}  (Sai số bình phương trung bình)")
    print(f"   R²   = {r2:.4f}  (Hệ số xác định - càng gần 1 càng tốt)")
    print(f"   MAPE = {mape:.2f}%   (Sai số phần trăm trung bình)")

In [ ]:
def cv_evaluate(name, model_class, model_params, X, y, n_splits=5, lo=0, hi=500):
    """
    Cross-validation theo thời gian
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_results = []

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X), 1):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        model = model_class(**model_params)
        model.fit(X_tr, y_tr)

        pred_raw = model.predict(X_te)
        pred_clipped = np.clip(pred_raw, lo, hi)

        r2 = r2_score(y_te, pred_clipped)
        mae = mean_absolute_error(y_te, pred_clipped)
        rmse = np.sqrt(mean_squared_error(y_te, pred_clipped))

        fold_results.append(
            {
                "Model": name,
                "Fold": fold,
                "n_train": len(tr_idx),
                "n_test": len(te_idx),
                "R²": round(r2, 4),
                "MAE": round(mae, 2),
                "RMSE": round(rmse, 2),
                "Pred_min_raw": round(pred_raw.min(), 1),
                "Pred_max_raw": round(pred_raw.max(), 1),
            }
        )

    return pd.DataFrame(fold_results)


def cv_summary(df_folds):
    """Tổng hợp Mean ± Std cho mỗi model"""
    return (
        df_folds.groupby("Model")
        .agg(
            R2_mean=("R²", "mean"),
            R2_std=("R²", "std"),
            MAE_mean=("MAE", "mean"),
            RMSE_mean=("RMSE", "mean"),
        )
        .round(4)
        .reset_index()
    )

### **3.1. Ridge Regression (Baseline)**

Ridge Regression là mô hình hồi quy tuyến tính có thêm L2 Regularization.

Mô hình này được dùng làm **Baseline** - đơn giản, huấn luyện nhanh giúp đánh giá xem các mô hình phức tạp hơn có thực sự cải thiện không.

In [ ]:
# Định nghĩa mô hình Ridge Regression
ridge_reg = Ridge(
    alpha=10.0,  # Hệ số L2 regularization - càng lớn càng tránh overfit
    fit_intercept=True,  # Có tính hệ số chặn hay không
    max_iter=1000,  # Số vòng lặp tối đa
    random_state=42,
)

# Train mô hình
ridge_reg.fit(X_train, y_train_reg)

In [ ]:
# Dự đoán và đánh giá
y_pred_ridge = ridge_reg.predict(X_test)
y_pred_ridge = np.clip(y_pred_ridge, 0, 500)

evaluate("Ridge Regression", y_test_reg, y_pred_ridge)

#### **Đánh giá các metrics:**

- **R² = 0.7491:** Mô hình giải thích được khoảng 75% sự biến thiên của dữ liệu. Đây là con số ấn tượng với mô hình tuyến tính đơn giản - cho thấy các feature có mối quan hệ tương đối tuyến tính với AQI ngày mai (Trong các bài toán thực tế, mức R² > 0.7 được đánh giá là một mô hình tốt và có độ tin cậy cao).

- **MAPE = 8.86%:** Dự đoán của mô hình chỉ lệch khoảng 9% so với con số thực tế - tương đương độ chính xác ~91%, khá tốt cho một mô hình Baseline (Trong thực tế, một mô hình dự đoán có MAPE dưới 10% (độ chính xác > 90%) thường được coi là xuất sắc và có giá trị cao để ra quyết định).

- **MAE = 8.27:** Trung bình mỗi dự đoán lệch 8.26 đơn vị AQI so với thực tế.

- **Điểm cần lưu ý: RMSE = 10.78 cao hơn MAE** → mô hình có một số dự đoán sai lệch lớn, đặc biệt vào những ngày ô nhiễm nặng mà mô hình tuyến tính khó nắm bắt. Nhưng khoảng cách giữa RMSE và MAE cũng tương đối hẹp (chênh lệch khoảng 2.51), điều này chứng tỏ mô hình rất ổn định và không tạo ra nhiều lỗi dự đoán nghiêm trọng.

> **Tóm lại:** Tại ngưỡng tối ưu alpha = 10, mô hình Ridge Regression đạt độ tin cậy cao với tỷ lệ giải thích dữ liệu (R²) lên tới 74.91%. Sai số dự đoán trung bình (MAPE) được duy trì ở mức thấp, dưới 9%. Đặc biệt, khoảng cách hẹp giữa chỉ số RMSE (10.78) và MAE (8.27) cho thấy mô hình hoạt động vô cùng ổn định, kiểm soát tốt hiện tượng nhiễu và hạn chế tối đa các dự đoán sai lệch lớn.

In [ ]:
# Xem hệ số (coefficients)
coef_ridge = (
    pd.Series(ridge_reg.coef_, index=X_train.columns)
    .sort_values(key=abs, ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
coef_ridge[::-1].plot(kind="barh", color="#3498DB", alpha=0.85)
plt.xlabel("Coefficient Value")
plt.title("Ridge - Top 20 Hệ số (|coef| lớn nhất)")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "ridge_coefficients")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()


#### **Kiểm tra Ridge bằng Cross-Validation**

**Mục đích cốt lõi của Cross-Validation (Xác thực chéo)** là để đánh giá chính xác và khách quan nhất khả năng dự đoán của mô hình trên dữ liệu mới (dữ liệu mà nó chưa từng nhìn thấy), từ đó giúp đảm bảo mô hình học được "quy luật thực sự" thay vì chỉ "học vẹt" trên tập dữ liệu huấn luyện.

In [ ]:
df_cv_ridge = cv_evaluate(
    "Ridge Regression", Ridge, {"alpha": 10}, X_train, y_train_reg
)
display(df_cv_ridge)
display(cv_summary(df_cv_ridge))


**Đánh giá:**

- **Ridge (alpha=10):** R² Mean = 0.70 ± 0.10

- **Sau khi tăng alpha từ giá trị mặc định lên 10.0, Ridge đạt kết quả
ổn định:** R² CV Mean = 0.70 ± 0.10, không còn fold nào cho giá trị
âm hoặc dự đoán AQI âm (Pred_min luôn dương qua tất cả 5 folds).

- **So sánh với kết quả Test set đơn (R² = 0.7491):** CV Mean = 0.70 gần
tương đương, cho thấy con số 0.7491 không phải ăn may mà phản
ánh đúng năng lực thực sự của Ridge khi đã lựa chọn alpha hợp lý.

#### **Kết luận mô hình Ridge Regression:**

- **Đánh giá tổng thể:** Mô hình Baseline đạt kết quả tốt hơn kỳ vọng - cho thấy AQI ngày mai có mối quan hệ tuyến tính đáng kể với các feature hiện tại.

- **Kết quả: MAE = 8.27 | RMSE = 10.78 | R² = 0.7491 | MAPE = 8.86%**

**Đánh giá:** R² = 0.7491 cho thấy các feature có mối quan hệ tuyến tính khá tốt với AQI ngày mai - đặc biệt các feature như pm2_5, là feature ảnh hưởng lớn nhất tới mô hình

- **Ý nghĩa thực tiễn:** MAE = 8.27 nghĩa là sai số dự đoán AQI trung bình là ±8.27 đơn vị - chấp nhận được cho bài toán dự đoán môi trường. MAPE = 8.86% cho thấy model dự đoán sai ~9% so với giá trị thực.

- **Giới hạn:** Là mô hình tuyến tính nên không nắm bắt được các pattern phi tuyến - đặc biệt vào những ngày AQI biến động bất thường do thời tiết cực đoan.

- **Hướng cải thiện:** Thử các mô hình khác.

> **Kết luận:** Ridge với alpha = 10 là một ứng viên ĐÁNG TIN CẬY cho Best Model, cần so sánh tiếp với CV của Decision Tree, Random Forest,
LightGBM, XGBoost để đưa ra quyết định cuối cùng.


In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "ridge_regressor.pkl")
joblib.dump(ridge_reg, save_path)
print("Đã lưu ridge_regressor.pkl")

### **3.2. Decision Tree**

Decision Tree là mô hình cây quyết định đơn giản nhất - chia nhỏ tập dữ liệu thành các nhóm đồng nhất dựa trên các điều kiện (luật "Nếu - Thì"). Thường dùng làm baseline cho các mô hình ensemble (Random Forest, XGBoost, LightGBM).

Mô hình này có khả năng tự động tìm ra các mối quan hệ phi tuyến tính phức tạp giữa các chỉ số ô nhiễm và chỉ số AQI ngày mai mà không cần giả định dữ liệu tuân theo phân phối nào.

**Lưu ý:** Với dataset chỉ ~1013 mẫu Train, Decision Tree rất dễ overfitting nếu để `max_depth` quá sâu - cần giới hạn chặt.

In [ ]:
# Định nghĩa mô hình Decision
dt_reg = DecisionTreeRegressor(
    max_depth=3,  # Giới hạn thấp - dataset nhỏ
    min_samples_leaf=30,  # Mỗi lá phải có tối thiểu 30 mẫu
    random_state=42,
)

# Train mô hình
dt_reg.fit(X_train, y_train_reg)


In [ ]:
# Dự đoán và đánh giá
y_pred_dt = dt_reg.predict(X_test)
y_pred_dt = np.clip(y_pred_dt, 0, 500)

evaluate("Decision Tree Regressor", y_test_reg, y_pred_dt)

#### **Đánh giá các metrics:**

- **R² = 0.5675:** Mô hình chỉ giải thích được khoảng 56.75% sự biến thiên của dữ liệu. Đây là một con số ở mức trung bình. Nó cho thấy cấu trúc hiện tại của Decision Tree (có thể do bị giới hạn độ sâu và số lượng mẫu ở lá) chưa đủ phức tạp để nắm bắt toàn bộ xu hướng của dữ liệu AQI, dẫn đến hiệu suất thấp hơn rõ rệt so với mô hình tuyến tính (đạt ~75%).

- **MAPE = 10.51%:** Dự đoán của mô hình lệch khoảng 10.5% so với thực tế. Độ chính xác tương đương khoảng 89.5% là một mức có thể chấp nhận được, nhưng đã vượt qua ngưỡng ổn định trong thực tế (<10%).

- **MAE = 10.21:** Trung bình mỗi dự đoán lệch 10.21 đơn vị AQI so với thực tế.

- **Điểm cần lưu ý: RMSE = 14.15 cao hơn khá nhiều so với MAE** → Khoảng chênh lệch (gần 4 đơn vị) chỉ ra rằng mô hình đang mắc phải một số sai số dự đoán rất lớn. Nguyên nhân xuất phát từ bản chất của Decision Tree: Các dự đoán ở nút lá chỉ là trung bình cộng của các mẫu trong lá đó. Khi gặp những ngày có AQI cực kỳ cao, mô hình thường dự đoán thấp hơn thực tế rất nhiều, khiến sai số bị bình phương lên trong chỉ số RMSE.

> **Tóm lại:** Khác với sự ổn định của Ridge Regression, mô hình Decision Tree hiện tại chỉ đạt hiệu suất ở mức trung bình thấp với tỷ lệ giải thích dữ liệu (R²) xấp xỉ 57%. Sai số phần trăm (MAPE) ở mức hơn 10% và khoảng cách chênh lệch rõ ràng giữa RMSE và MAE cho thấy mô hình khá nhạy cảm với dữ liệu và gặp khó khăn lớn trong việc dự đoán chính xác những ngày có mức độ ô nhiễm đột biến. Mức độ tin cậy của mô hình này chưa cao để có thể đưa vào sử dụng thực tế.

In [ ]:
# So sánh R² Train vs Test - Kiểm tra overfitting
r2_train = r2_score(y_train_reg, dt_reg.predict(X_train))
r2_test = r2_score(y_test_reg, y_pred_dt)
print(
    f"R² Train = {r2_train:.4f} | R² Test = {r2_test:.4f} | Gap = {r2_train - r2_test:.4f}"
)

In [ ]:
# Feature Importance
fi_dt = (
    pd.Series(dt_reg.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
fi_dt[::-1].plot(kind="barh", color="#9B59B6", alpha=0.85)
plt.xlabel("Importance Score")
plt.title("Decision Tree - Top 20 Feature Importance")
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "dt_feature_importance")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()


#### **Kiểm tra Decision Tree bằng Cross-Validation**

In [ ]:
df_cv_dt = cv_evaluate(
    "Decision Tree",
    DecisionTreeRegressor,
    {"max_depth": 3, "min_samples_leaf": 30, "random_state": 42},
    X_train,
    y_train_reg,
)
display(df_cv_dt)
display(cv_summary(df_cv_dt))


**So sánh với Baseline:**

- **Decision Tree:** R² Mean = 0.65 ± 0.11

- **Sau Feature Engineering kỹ lưỡng (lag, rolling, sin/cos),** quan hệ giữa các đặc trưng và biến mục tiêu đã trở nên gần tuyến tính. Với dataset có kích thước hạn chế (chỉ ~1013 mẫu Train), Decision Tree với độ phức tạp cao hơn (cây quyết định, nhiều tham số) dễ bị nhiễu hơn so với Ridge - một mô hình tuyến tính đơn giản với regularization phù hợp.

- **So sánh với kết quả Test set đơn (R² = 0.5675):** CV Mean = 0.65 lệch khá nhiều, cho thấy con số 0.5675 không phản ánh đúng năng lực thực sự của Decision Tree (Bị overfit khá nặng).

- **Đây là minh chứng cho nguyên lý "Occam's Razor":** Mô hình đơn giản
không nhất thiết kém hơn, đặc biệt khi dữ liệu đã qua Feature
Engineering tốt.

#### **Kết luận Decision Tree:**

- **Đánh giá tổng thể:** Mô hình đạt kết quả thấp so với kỳ vọng - hiệu suất mô hình đạt mức thấp.

- **Kết quả: MAE = 10.21 | RMSE = 14.15 | R² = 0.5675 | MAPE = 10.51%**

- **Ý nghĩa thực tiễn:** MAE = 10.21 nghĩa là sai số dự đoán AQI trung bình là ±10.21 đơn vị - sai lệch nghiêm trọng cho bài toán dự đoán môi trường. MAPE = 10.51% cho thấy model dự đoán sai ~11% so với giá trị thực.

- **Giới hạn:** Là mô hình cây quyết định đơn giản nhất nên khả năng sai lệch cao - đặc biệt vào những ngày AQI biến động bất thường do thời tiết cực đoan.

- **Hướng cải thiện:** Thử các mô hình cây quyết định mạnh mẽ khác.

> **Kết luận:** Có thể là mô hình tệ nhất, không bao gồm SARIMA.


In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "dt_regressor.pkl")
joblib.dump(dt_reg, save_path)
print("Đã lưu dt_regressor.pkl")

### **3.3. Random Forest**

In [ ]:
# Định nghĩa mô hình Random Forest
rf_reg = RandomForestRegressor(
    n_estimators=500,  # Tăng số lượng cây để kết quả dự đoán ổn định hơn
    max_depth=12,  # Độ sâu tối đa được tăng nhẹ vì đã có lá kiểm soát
    min_samples_split=5,  # Tối thiểu 5 mẫu để tiếp tục tách node
    min_samples_leaf=2,  # Tối thiểu 2 mẫu tại một lá cuối cùng (chống học vẹt nhiễu)
    max_features="sqrt",  # Chỉ dùng căn bậc 2 số features khi rẽ nhánh (chống đa cộng tuyến)
    random_state=42,  # Cố định state để tái lập kết quả
    n_jobs=-1,  # Dùng tất cả luồng CPU
)

# Train mô hình
rf_reg.fit(X_train, y_train_reg)

In [ ]:
# Dự đoán và đánh giá
y_pred_rf = rf_reg.predict(X_test)
y_pred_rf = np.clip(y_pred_rf, 0, 500)

evaluate("Random Forest Regressor", y_test_reg, y_pred_rf)

#### **Đánh giá các metrics:**

- **MAE = 9.43:** Trung bình mỗi dự đoán của mô hình lệch khoảng 9.43 đơn vị so với giá trị thực tế. Đây là mức sai số tuyệt đối khá thấp, cho thấy mô hình có độ chuẩn xác ổn định trên phần lớn các điểm dữ liệu.
- **RMSE = 13.23:** Giá trị RMSE lớn hơn MAE ($13.23 > 9.43$) chứng tỏ trong tập dữ liệu vẫn xuất hiện một số điểm dự đoán có sai số lớn (outliers). Tuy nhiên, khoảng cách giữa hai chỉ số này không quá biệt lập, cho thấy mô hình kiểm soát nhiễu tương đối tốt.
- **MSE = 174.96:** Đo lường trung bình bình phương các sai số, được sử dụng làm hàm mất mát (loss function) chính trong quá trình tối ưu. Chỉ số này nhạy cảm với các sai số lớn nhằm phạt nặng các cú "đoán mò" của mô hình.
- **R² = 0.6221 (62.21%):** Mô hình giải thích được khoảng 62.21% sự biến động của biến mục tiêu dựa trên các đặc trưng đầu vào. Đối với một bài toán hồi quy phi tuyến có độ nhiễu cao, đây là một mức hiệu năng tương đối ổn định (bắt đầu đạt ngưỡng khá tốt).
- **MAPE = 9.79%:** Sai số phần trăm trung bình chỉ ở mức 9.79% (dưới 10%). Điều này khẳng định mô hình có độ tin cậy và độ chính xác cao khi áp dụng vào thực tế, sai số lệch không đáng kể so với quy mô của giá trị cần dự đoán.

In [ ]:
# Feature Importance
fi = (
    pd.Series(rf_reg.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
fi[::-1].plot(kind="barh", color="#27AE60", alpha=0.85)
plt.xlabel("Importance Score")
plt.title("Random Forest - Top 20 Feature Importance", fontweight="bold")

plt.grid(False)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "rf_feature_importance")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Kiểm tra Random Forest bằng Cross-Validation**

In [ ]:
df_cv_rf = cv_evaluate(
    "Random Forest",
    RandomForestRegressor,
    {
        "n_estimators": 500,
        "max_depth": 12,
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "random_state": 42,
        "n_jobs": -1,
    },
    X_train,
    y_train_reg,
)

display(df_cv_rf)
display(cv_summary(df_cv_rf))

#### **Kết luận tổng thể mô hình Random Forest**

Nhìn lại toàn bộ quá trình đánh giá, có thể khẳng định mô hình Random Forest đã hoàn thành xuất sắc vai trò của mình và chứng minh được tính hiệu quả thực tiễn:

- **Cấu hình tối ưu & Tính ổn định cao:** Bộ tham số định hướng  (`max_depth=12`, `min_samples_leaf=2`, `max_features='sqrt'`) đã phát huy tối đa tác dụng. Mô hình đạt độ ổn định vượt trội qua các nếp gấp thời gian ($R^2 \text{ Mean} = 0.67 \pm 0.05$), khắc phục hoàn toàn điểm yếu dễ bị trồi sụt của cây quyết định đơn lẻ (Decision Tree) và hoàn toàn không bị overfitting.
- **Khả năng nắm bắt quy luật sâu sắc:** Thuật toán đã bóc tách thành công các lõi ô nhiễm tại TP.HCM: từ sự thống trị của nhóm bụi mịn (PM2.5, PM10), khí thải giao thông ($NO_2, CO$), cho đến việc tận dụng triệt để "quán tính" của ô nhiễm thông qua các biến phái sinh thời gian (`t-1`, `rolling_max_7`).
- **Độ tin cậy trong dự báo:** Với mức sai số phần trăm (MAPE) dưới 10% trên Test set và sai số tuyệt đối trung bình (MAE) quanh ngưỡng **8.49** khi kiểm định chéo, các dự báo đưa ra rất sát với thực tế. Dù mô hình vẫn còn hụt nhịp nhẹ trước các điểm nghẽn dị thường hoặc biến động thời tiết cực đoan (thể hiện qua sự chênh lệch của RMSE và kết quả tại Fold 5), đây vẫn là mức hiệu năng cực kỳ ấn tượng cho một bài toán môi trường phức tạp.

> **Tóm lại:** Random Forest đã thiết lập thành công một **Baseline phi tuyến tính kiên cố và đáng tin cậy**. Nó không chỉ cung cấp những insight đắt giá về mặt dữ liệu mà còn tạo ra một thước đo chuẩn mực, tạo đà hoàn hảo để chúng ta tự tin bước vào việc tinh chỉnh các thuật toán Gradient Boosting mạnh mẽ hơn ở giai đoạn tiếp theo.

In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "rf_regressor.pkl")
joblib.dump(rf_reg, save_path)

print("Đã lưu rf_regressor.pkl")

### **3.4. LightGBM - XGBoost**

#### **LightGBM**

LightGBM là một framework học máy dựa trên thuật toán cây quyết định và kỹ thuật Gradient Boosting.

Mô hình này được thiết kế cực kỳ tối ưu về tốc độ huấn luyện và lượng bộ nhớ tiêu thụ, giúp nó xử lý được các tập dữ liệu khổng lồ một cách trơn tru.

In [ ]:
# Định nghĩa mô hình LightGBM
lgbm_reg = lgb.LGBMRegressor(
    n_estimators=500,  # Số cây - nhiều hơn thì chính xác hơn, Train lâu hơn
    learning_rate=0.05,  # Tốc độ học - nhỏ thì ổn định hơn nhưng cần nhiều cây hơn
    max_depth=6,  # Độ sâu tối đa mỗi cây - tránh overfitting
    num_leaves=31,  # Số lá - LightGBM dùng leaf-wise, không dùng depth-wise
    min_child_samples=20,  # Số mẫu tối thiểu mỗi lá - tránh overfitting
    subsample=0.8,  # Dùng 80% dữ liệu mỗi cây - tăng tính đa dạng
    subsample_freq=1,  # Thực hiện lấy mẫu lại 80% data ở mỗi cây
    colsample_bytree=0.8,  # Dùng 80% features mỗi cây
    reg_alpha=0.1,  # L1 regularization
    reg_lambda=0.1,  # L2 regularization
    random_state=42,
    verbose=-1,  # Tắt log khi train
    n_jobs=-1,  # Dùng tất cả CPU
)

In [ ]:
# Train mô hình
lgbm_reg.fit(
    X_train,
    y_train_reg,
    eval_set=[
        (X_train, y_train_reg),
        (X_test, y_test_reg),
    ],  # Theo dõi loss trên Train và Test mỗi 50 cây
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50, verbose=False
        ),  # Dừng sớm nếu không cải thiện
        lgb.log_evaluation(period=100),  # In log mỗi 100 cây
    ],
)

print(f"Best iteration: {lgbm_reg.best_iteration_}")

In [ ]:
# Dự đoán và đánh giá
y_pred_lgbm = lgbm_reg.predict(X_test)
y_pred_lgbm = np.clip(y_pred_lgbm, 0, 500)  # AQI trong [0, 500]

evaluate("LightGBM Regressor", y_test_reg, y_pred_lgbm)

**Đánh giá các metrics:**

- **MAPE = 9.58%:** Điểm sáng nhất, dự đoán của mô hình chỉ lệch khoảng gần 10% so với con số thực tế (tương đương độ chính xác ~90%).

- **R² = 0.6760:** Mô hình đã nắm bắt và giải thích được khoảng 68% sự biến thiên của dữ liệu. Chỉ còn khoảng 32% là do các yếu tố ngẫu nhiên hoặc các features mà mô hình chưa được cung cấp. Với việc dự đoán chỉ số chất lượng không khí (được xem là phức tạp), thì 68% là một con số rất đáng tin cậy (khi mà bộ dữ liệu thiếu những tác động bên ngoài).

- **Điểm cần lưu ý:** MAE = 9.20 (đa số các dự đoán của bạn sẽ lệch khoảng 9.20 đơn vị so với thực tế) và RMSE = 12.25 (trong tập Test, có một vài mẫu bị mô hình dự đoán sai lệch cực kỳ xa).

In [ ]:
# Feature Importance của LightGBM
fi_lgbm = (
    pd.Series(lgbm_reg.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
fi_lgbm[::-1].plot(kind="barh", color="#27AE60", alpha=0.85)
plt.xlabel("Importance Score")
plt.title("LightGBM - Top 20 Feature Importance", fontweight="bold")

plt.grid(False)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "lgbm_feature_importance")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

**Kiểm tra LightGBM bằng Cross-Validation**

In [ ]:
df_cv_lgbm = cv_evaluate(
    "LightGBM",
    lgb.LGBMRegressor,
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 6,
        "random_state": 42,
        "verbose": -1,
    },
    X_train,
    y_train_reg,
)
display(df_cv_lgbm)
display(cv_summary(df_cv_lgbm))


**Kết luận mô hình LightGBM Regressor:**

- **Đánh giá tổng thể:** Mô hình đạt kết quả ở mức trung bình khá - chưa phải kết quả tốt nhất có thể đạt được với dataset này.

- **Kết quả: MAE = 9.20 | RMSE = 12.25 | R² = 0.6760 | MAPE = 9.58%**

- **Đánh giá:** R² = 0.6760 phản ánh đúng độ khó của bài toán - không phải do
pipeline xử lý kém. AQI tại TPHCM có tính ngẫu nhiên cao do:
1. Mưa nhiệt đới đột ngột làm AQI giảm mạnh không theo quy luật
2. Dataset chỉ có chỉ số ô nhiễm, thiếu các biến thời tiết quan
trọng như nhiệt độ, độ ẩm, tốc độ gió - những yếu tố ảnh hưởng
trực tiếp đến khả năng phân tán bụi.

- **Ý nghĩa thực tiễn:** MAE = 9.20 nghĩa là sai số dự đoán AQI trung bình là ±9.20 đơn vị. Với thang AQI chia theo mức 50 đơn vị. MAPE = 9.58% cho thấy model dự đoán sai ~10% so với giá trị thực.

- **Giới hạn:** Model sai nhiều nhất vào tháng 6 (chuyển mùa mưa) khi AQI biến
động bất thường. Hướng cải thiện: bổ sung dữ liệu thời tiết (nhiệt độ, độ ẩm, lượng mưa, tốc độ gió) từ OpenWeatherMap API.

- **Vấn đề:** Mô hình có dấu hiệu overfit nghiêm trọng.

In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "lgbm_regressor.pkl")
joblib.dump(lgbm_reg, save_path)
print("Đã lưu lgbm_regressor.pkl")

#### **XGBoost**

XGBoost là một trong những thuật toán học máy dựa trên cây quyết định mạnh mẽ nhất hiện nay.

Khác với LightGBM phát triển theo lá (leaf-wise), XGBoost phát triển cây theo từng tầng (depth-wise). Điều này giúp mô hình kiểm soát tốt hơn kiến trúc của cây, tuy nhiên tốc độ huấn luyện có thể chậm hơn LightGBM một chút khi xử lý dữ liệu lớn.

In [ ]:
# Định nghĩa mô hình LightGBM
lgbm_reg = lgb.LGBMRegressor(
    n_estimators=500,  # Số cây - nhiều hơn thì chính xác hơn, Train lâu hơn
    learning_rate=0.05,  # Tốc độ học - nhỏ thì ổn định hơn nhưng cần nhiều cây hơn
    max_depth=6,  # Độ sâu tối đa mỗi cây - tránh overfitting
    num_leaves=31,  # Số lá - LightGBM dùng leaf-wise, không dùng depth-wise
    min_child_samples=20,  # Số mẫu tối thiểu mỗi lá - tránh overfitting
    subsample=0.8,  # Dùng 80% dữ liệu mỗi cây - tăng tính đa dạng
    subsample_freq=1,  # Thực hiện lấy mẫu lại 80% data ở mỗi cây
    colsample_bytree=0.8,  # Dùng 80% features mỗi cây
    reg_alpha=0.1,  # L1 regularization
    reg_lambda=0.1,  # L2 regularization
    random_state=42,
    verbose=-1,  # Tắt log khi train
    n_jobs=-1,  # Dùng tất cả CPU
)

In [ ]:
# Train mô hình
lgbm_reg.fit(
    X_train,
    y_train_reg,
    eval_set=[
        (X_train, y_train_reg),
        (X_test, y_test_reg),
    ],  # Theo dõi loss trên Train và Test mỗi 50 cây
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50, verbose=False
        ),  # Dừng sớm nếu không cải thiện
        lgb.log_evaluation(period=100),  # In log mỗi 100 cây
    ],
)

print(f"Best iteration: {lgbm_reg.best_iteration_}")

In [ ]:
# Dự đoán và đánh giá
y_pred_lgbm = lgbm_reg.predict(X_test)
y_pred_lgbm = np.clip(y_pred_lgbm, 0, 500)  # AQI trong [0, 500]

evaluate("LightGBM Regressor", y_test_reg, y_pred_lgbm)

**Đánh giá các metrics:**

- **MAPE = 9.58%:** Điểm sáng nhất, dự đoán của mô hình chỉ lệch khoảng gần 10% so với con số thực tế (tương đương độ chính xác ~90%).

- **R² = 0.6760:** Mô hình đã nắm bắt và giải thích được khoảng 68% sự biến thiên của dữ liệu. Chỉ còn khoảng 32% là do các yếu tố ngẫu nhiên hoặc các features mà mô hình chưa được cung cấp. Với việc dự đoán chỉ số chất lượng không khí (được xem là phức tạp), thì 68% là một con số rất đáng tin cậy (khi mà bộ dữ liệu thiếu những tác động bên ngoài).

- **Điểm cần lưu ý:** MAE = 9.20 (đa số các dự đoán của bạn sẽ lệch khoảng 9.20 đơn vị so với thực tế) và RMSE = 12.25 (trong tập Test, có một vài mẫu bị mô hình dự đoán sai lệch cực kỳ xa).

In [ ]:
# Feature Importance của LightGBM
fi_lgbm = (
    pd.Series(lgbm_reg.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
fi_lgbm[::-1].plot(kind="barh", color="#27AE60", alpha=0.85)
plt.xlabel("Importance Score")
plt.title("XGBoost - Top 20 Feature Importance", fontweight="bold")

plt.grid(False)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "xgb_feature_importance")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

**Kiểm tra LightGBM bằng Cross-Validation**

In [ ]:
df_cv_lgbm = cv_evaluate(
    "LightGBM",
    lgb.LGBMRegressor,
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 6,
        "random_state": 42,
        "verbose": -1,
    },
    X_train,
    y_train_reg,
)
display(df_cv_lgbm)
display(cv_summary(df_cv_lgbm))


**Kết luận mô hình LightGBM Regressor:**

- **Đánh giá tổng thể:** Mô hình đạt kết quả ở mức trung bình khá - chưa phải kết quả tốt nhất có thể đạt được với dataset này.

- **Kết quả: MAE = 9.20 | RMSE = 12.25 | R² = 0.6760 | MAPE = 9.58%**

- **Đánh giá:** R² = 0.6760 phản ánh đúng độ khó của bài toán - không phải do
pipeline xử lý kém. AQI tại TPHCM có tính ngẫu nhiên cao do:
1. Mưa nhiệt đới đột ngột làm AQI giảm mạnh không theo quy luật
2. Dataset chỉ có chỉ số ô nhiễm, thiếu các biến thời tiết quan
trọng như nhiệt độ, độ ẩm, tốc độ gió - những yếu tố ảnh hưởng
trực tiếp đến khả năng phân tán bụi.

- **Ý nghĩa thực tiễn:** MAE = 9.20 nghĩa là sai số dự đoán AQI trung bình là ±9.20 đơn vị. Với thang AQI chia theo mức 50 đơn vị. MAPE = 9.58% cho thấy model dự đoán sai ~10% so với giá trị thực.

- **Giới hạn:** Model sai nhiều nhất vào tháng 6 (chuyển mùa mưa) khi AQI biến
động bất thường. Hướng cải thiện: bổ sung dữ liệu thời tiết (nhiệt độ, độ ẩm, lượng mưa, tốc độ gió) từ OpenWeatherMap API.

- **Vấn đề:** Mô hình có dấu hiệu overfit nghiêm trọng.

In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "lgbm_regressor.pkl")
joblib.dump(lgbm_reg, save_path)
print("Đã lưu lgbm_regressor.pkl")

In [ ]:
# Định nghĩa mô hình XGBoost
xgb_reg = xgb.XGBRegressor(
    n_estimators=500,  # Số cây tối đa
    learning_rate=0.05,  # Tốc độ học
    max_depth=6,  # Độ sâu tối đa mỗi cây (depth-wise)
    min_child_weight=20,  # Số mẫu tối thiểu ở node lá
    subsample=0.8,  # Lấy ngẫu nhiên 80% dữ liệu để train mỗi cây
    colsample_bytree=0.8,  # Lấy ngẫu nhiên 80% features cho mỗi cây
    reg_alpha=0.1,  # L1 regularization
    reg_lambda=0.1,  # L2 regularization
    random_state=42,
    n_jobs=-1,  # Tận dụng tối đa CPU
    objective="reg:squarederror",  # Hàm mất mát cho bài toán hồi quy
    early_stopping_rounds=50,  # Dừng sớm nếu không học nữa
)

In [ ]:
# Train mô hình
xgb_reg.fit(
    X_train,
    y_train_reg,
    eval_set=[
        (X_train, y_train_reg),
        (X_test, y_test_reg),
    ],  # Theo dõi cả Train và Test
    verbose=100,  # In log mỗi 100 cây
)

print(f"Best iteration: {xgb_reg.best_iteration}")

In [ ]:
# Dự đoán và giới hạn giá trị AQI trong khoảng [0, 500]
y_pred_xgb = xgb_reg.predict(X_test)
y_pred_xgb = np.clip(y_pred_xgb, 0, 500)

evaluate("XGBoost Regressor", y_test_reg, y_pred_xgb)

**Đánh giá các metrics:**

- **MAPE = 9.57%:** Sai số phần trăm trung bình ở mức rất tốt (chỉ lệch khoảng ~9.57% so với thực tế). Ngang ngửa với LightGBM (9.58%).

- **R² = 0.6841:** Mô hình nắm bắt và giải thích được khoảng 68.41% sự biến thiên của dữ liệu AQI. Một kết có phần tốt LightGBM (67.60%).

- **MAE = 9.14 & MSE = 146.26:** Sai số tuyệt đối trung bình là 9.14. Mức chênh lệch giữa MAE và RMSE (quy đổi khoảng 12.09) (tương đương với LightGBM) cho thấy thuật toán phân chia theo tầng (depth-wise) của XGBoost thỉnh thoảng vẫn gặp khó khăn trong việc dự đoán những ngày có chỉ số ô nhiễm cao bất thường.

In [ ]:
# Feature Importance của XGBoost
fi_xgb = (
    pd.Series(xgb_reg.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))
fi_xgb[::-1].plot(kind="barh", color="#E67E22", alpha=0.85)
plt.xlabel("Importance Score")
plt.title("XGBoost - Top 20 Feature Importance", fontweight="bold")

plt.grid(False)
plt.tight_layout()
plt.show()

print("Top 10 features quan trọng nhất (XGBoost):")
for feat, score in fi_xgb.head(10).items():
    print(f"  {feat:<35} {score:.4f}")

**Kiểm tra XGBoost bằng Cross-Validation**

In [ ]:
df_cv_xgb = cv_evaluate(
    "XGBoost",
    xgb.XGBRegressor,
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 6,
        "min_child_weight": 20,
        "random_state": 42,
        "verbosity": 0,
    },
    X_train,
    y_train_reg,
)
display(df_cv_xgb)
display(cv_summary(df_cv_xgb))

**Đánh giá tổng thể và Kết luận mô hình XGBoost:**

- **Kết quả: MAE = 9.14 | RMSE = 12.09 | R² = 0.6841 | MAPE = 9.57%**

- **Về Đặc trưng (Feature Importance):** Biểu đồ thể hiện cực kỳ rõ rệt đặc tính của XGBoost khi phân bổ trọng số: Chỉ số bụi mịn `pm2_5` đóng vai trò xương sống và chiếm ưu thế tuyệt đối (điểm Gain khoảng 0.35), bỏ xa tất cả các biến còn lại. Theo sau là `pm10` và `carbon_monoxide`. Điều này cho thấy mô hình XGBoost ra quyết định phụ thuộc cực kỳ nặng nề vào các chỉ số ô nhiễm cốt lõi và nồng độ khí thải trực tiếp, thay vì dựa vào các biến xu hướng chuỗi thời gian (như `t-1`, `t-2`).

- **Đánh giá thuật toán:** Mặc dù nắm bắt rất tốt nồng độ chất gây ô nhiễm, việc XGBoost bị quá phụ thuộc vào `pm2_5` khiến nó có phần cứng nhắc. Khi có các yếu tố ngoại cảnh gây thay đổi AQI đột ngột (như mưa rào hoặc gió lớn làm loãng bụi nhưng pm2_5 đo được trước đó vẫn cao), mô hình sẽ lúng túng và dễ dự đoán sai do các biến thời tiết không có đủ trọng lượng để can thiệp vào các tầng quyết định (depth-wise) của cây.

- **Hướng triển khai thực tiễn:** Ở bước tiếp theo, nhóm sẽ áp dụng kỹ thuật Ensemble (Kết hợp mô hình): Lấy sự ổn định, bám sát các chỉ số hóa học của XGBoost kết hợp với độ nhạy bén, linh hoạt xử lý chuỗi thời gian của LightGBM để bù trừ khuyết điểm, từ đó tạo ra một mô hình dự đoán toàn diện và chống chịu tốt hơn với các điểm bất thường.

In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "xgb_regressor.pkl")
joblib.dump(xgb_reg, save_path)
print("Đã lưu xgb_regressor.pkl")

#### **Kết hợp XGBoost và LightGBM**

In [ ]:
# 1. Thiết lập trọng số (Thử nhiều lần để tìm ra trọng số tốt nhất)
w_lgbm = 0.65
w_xgb = 0.35

# 2. Lấy dự đoán từ 2 mô hình
pred_lgbm = lgbm_reg.predict(X_test)
pred_xgb = xgb_reg.predict(X_test)

# 3. Giới hạn giá trị trong khoảng AQI thực tế [0, 500]
pred_lgbm = np.clip(pred_lgbm, 0, 500)
pred_xgb = np.clip(pred_xgb, 0, 500)

# 4. Trộn kết quả (Trung bình có trọng số)
y_pred_ensemble = (w_lgbm * pred_lgbm) + (w_xgb * pred_xgb)

# 5. Đánh giá kết quả
evaluate("Ensemble (65% LGBM + 35% XGB)", y_test_reg, y_pred_ensemble)

**Đánh giá các metrics:**

- **Mô hình hoạt động rất tốt cho nhu cầu dự báo:** MAPE = 9.52% và MAE = 9.13: Đây là hai con số tốt và ổn định. Việc mô hình chỉ sai số trung bình khoảng 9.52% (tương đương lệch ~9.5 đơn vị AQI) là một kết quả rất đáng mong đợi đối với dữ liệu môi trường vốn là hỗn loạn.

- **Sự đánh đổi chiến lược:** R² = 0.6822. Mô hình giải thích được 68.22% sự biến thiên của dữ liệu. Điểm số này có thể thấp hơn một chút xíu xiu so với lúc chạy XGBoost đơn lẻ (0.6841).

- **Vấn đề còn tồn đọng:**

**Khoảng cách giữa RMSE (12.13) và MAE (9.31).** Vì RMSE phạt rất nặng các điểm dự đoán sai lệch lớn, việc RMSE cao hơn MAE ~2.82 đơn vị cho thấy mô hình vẫn thỉnh thoảng bị sai lệch ở những ngày có AQI thay đổi đột biến (outliers), nhưng vẫn nằm trong mức độ cho phép.

**Bản chất Overfitting:** Dù Ensemble giúp mượt mà hóa kết quả dự đoán, nó không giải quyết tận gốc hiện tượng Overfitting của hai mô hình con. Mô hình hiện tại là sự kết hợp của hai mô hình tốt nhưng hay học vẹt.

> **Kết luận tổng thể:** Đã xây dựng được một mô hình Ensemble có tính thực tế rất cao (sai số dưới 10%). Mức trần hiệu suất R² ~ 68% hiện tại có lẽ là giới hạn tối đa mà bộ dữ liệu gốc (chỉ có chỉ số ô nhiễm, thiếu dữ liệu thời tiết) có thể cung cấp.

In [ ]:
print("Tỷ lệ tối ưu cho Ensemble")

# Danh sách các tỷ lệ (Trọng số LightGBM, Trọng số XGBoost)
test_weights = [(0.65, 0.35), (0.70, 0.30), (0.80, 0.20), (0.90, 0.10)]

for w_lgbm, w_xgb in test_weights:
    # Trộn dự đoán theo tỷ lệ
    pred_mix = (w_lgbm * pred_lgbm) + (w_xgb * pred_xgb)

    # Tính R² và MAPE
    r2 = r2_score(y_test_reg, pred_mix)
    mape = np.mean(np.abs((y_test_reg - pred_mix) / (y_test_reg + 1e-9))) * 100

    print(
        f"LGBM {w_lgbm * 100:.0f}% + XGB {w_xgb * 100:.0f}%  ->  R² = {r2:.4f}  |  MAPE = {mape:.2f}%"
    )

In [ ]:
samples = 50
plt.figure(figsize=(15, 6))

plt.plot(
    y_test_reg.values[:samples],
    label="Thực tế (Actual)",
    color="black",
    linewidth=2,
    marker="o",
)
plt.plot(
    pred_lgbm[:samples], label="LightGBM", color="#27AE60", linestyle="--", alpha=0.8
)
plt.plot(
    pred_xgb[:samples], label="XGBoost", color="#E67E22", linestyle="-.", alpha=0.8
)
plt.plot(y_pred_ensemble[:samples], label="Ensemble", color="#2980B9", linewidth=2)

plt.title(
    "So Sánh Dự Đoán AQI Của Các Mô Hình (50 Ngày Đầu Tập Test)",
    fontweight="bold",
    fontsize=14,
)
plt.xlabel("Days")
plt.ylabel("AQI")
plt.legend()
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "lgbm_xgb_aqi_compare")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

**Kết luận mô hình tốt nhất trong 3 mô hình này: XGBoost Regressor**

**Lý do:**

- **Hiệu suất thực tế nhỉnh hơn:** XGBoost dẫn đầu trên các thang đo quan trọng nhất của tập Test. Nó kiểm soát sai số lớn tốt hơn (RMSE thấp nhất: 12.09) và giải thích sự biến thiên của dữ liệu tốt nhất (R² cao nhất: 0.6841).

- **Tối ưu hóa tài nguyên triển khai:** Trong môi trường thực tế, việc chỉ duy trì và suy luận trên một mô hình XGBoost duy nhất sẽ giúp tiết kiệm đáng kể chi phí server, tăng tốc độ phản hồi và giảm thiểu rủi ro bảo trì hệ thống so với việc phải vận hành cồng kềnh cả hai mô hình (Ensemble).

**Hướng cải thiện vì LightGBM và XGBoost được cho là những mô hình tốt nhất:**

- **Bổ sung thêm dữ liệu (Ưu tiên hàng đầu):** Thuật toán hiện tại đã chạm trần sức mạnh nếu chỉ dùng dữ liệu AQI lịch sử. Bắt buộc phải tích hợp thêm API dữ liệu khí tượng: Tốc độ/hướng gió, lượng mưa, độ ẩm, và đặc biệt là chỉ số nghịch nhiệt (Inversion). Đây là chìa khóa để giải thích các cú tăng AQI đột biến.

- **Tối ưu hóa tham số tự động:**

**Giới hạn độ phức tạp của cây:** Giảm max_depth (xuống mức 3-5) và tăng min_child_weight để ngăn mô hình học thuộc lòng các điểm nhiễu (noise) lặt vặt trong chu kỳ bình thường.

**Tăng cường trừng phạt (Regularization):** Tinh chỉnh mạnh tay hơn các hệ số reg_alpha (L1) và reg_lambda (L2). Điều này sẽ ép mô hình loại bỏ các đặc trưng rác, giúp đường dự đoán trở nên cứng cáp, tổng quát hóa tốt hơn và sẵn sàng bắt các tín hiệu mạnh từ dữ liệu thời tiết mới được thêm vào.

In [ ]:
models_dir = os.path.join(ROOT, "models")
os.makedirs("models_dir", exist_ok=True)
save_path = os.path.join(models_dir, "ensemble_dict.pkl")

# Nhét tất cả vào một Dictionary
ensemble_package = {
    "model_lgbm": lgbm_reg,
    "model_xgb": xgb_reg,
    "w_lgbm": 0.65,
    "w_xgb": 0.35,
}
joblib.dump(ensemble_package, save_path)

print(f"Đã lưu thành công Gói Ensemble")

### **3.5. SARIMA**

SARIMA (Seasonal AutoRegressive Integrated Moving Average) là mô hình thống kê kinh điển cho Time Series, khác biệt hoàn toàn so với các model ML ở trên: Không dùng X_train (features đa biến) mà chỉ dùng lịch sử của chính biến mục tiêu (us_aqi).

Mô hình này học từ "quá khứ của chính AQI" (autoregressive) để dự đoán bước tiếp theo, đồng thời nắm bắt được tính mùa vụ (seasonality) nếu có pattern lặp lại theo chu kỳ.

**Mục đích đưa SARIMA vào so sánh:** Chứng minh giá trị của Machine Learning + Feature Engineering. Nếu SARIMA cho kết quả kém hơn rõ rệt so với các mô hình hồi quy, đó là bằng chứng cho thấy việc tạo ra 35-40 features (lag, rolling, sin/cos...) đã đóng góp thực sự vào độ chính xác, không phải chỉ là làm phức tạp hóa vấn đề không cần thiết.

In [ ]:
# Chuẩn bị dữ liệu cho SARIMA
## Lưu ý: us_aqi đã bị loại khỏi X_train/X_test (là TARGET gốc, model ML
## không được nhìn thấy) và cũng không có trong y_train/y_test (chỉ có
## target_reg_tomorrow đã shift). Series us_aqi gốc, liên tục, chưa scale
## được lấy lại từ data_processed.csv (xuất từ trước bước tạo X/y).

# Lọc us_aqi theo đúng index của X_train/X_test đã chia, đảm bảo SARIMA
# Train/Test trên cùng khoảng thời gian với các model ML
us_aqi_train = df_processed.loc[X_train.index, "us_aqi"]
us_aqi_test = df_processed.loc[X_test.index, "us_aqi"]

print(
    f"US AQI Train: {len(us_aqi_train)} ngày | {us_aqi_train.index.min().date()} -> {us_aqi_train.index.max().date()}"
)
print(
    f"US AQI Test : {len(us_aqi_test)} ngày  | {us_aqi_test.index.min().date()} -> {us_aqi_test.index.max().date()}"
)

In [ ]:
# Kiểm tra tính dừng (Stationarity) bằng ADF Test
adf_result = adfuller(us_aqi_train.dropna())

print("ADF Test - kiểm tra tính dừng trên us_aqi_train:")
print(f"   ADF Statistic = {adf_result[0]:.4f}")
print(f"   p-value       = {adf_result[1]:.4f}")
print(
    f"   => Series {'ĐÃ DỪNG (stationary)' if adf_result[1] <= 0.05 else 'CHƯA DỪNG, cần sai phân d = 1'}"
)

In [ ]:
# Định nghĩa mô hình SARIMA
# auto_arima tự tìm (p,d,q)(P,D,Q,m) tối ưu
sarima_model = auto_arima(
    us_aqi_train,
    seasonal=True,
    m=7,  # Chu kỳ mùa vụ (m = 12 (tháng), 7 (tuần) hoặc 365 (năm, tốn thời gian))
    d=0,  # Tận dụng kết quả chuỗi đã dừng từ ADF Test (Bậc sai phân bằng 0)
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
    max_p=3,
    max_q=3,
    max_P=2,
    max_Q=2,
    max_D=1,
    information_criterion="aic",
)

print(sarima_model.summary())

In [ ]:
# Train mô hình
n_steps = len(us_aqi_test)
sarima_predictions = sarima_model.predict(n_periods=n_steps)
print("Dự báo SARIMA 5 ngày đầu tiên trên tập Test:")
print(sarima_predictions.head())

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    us_aqi_test.index, us_aqi_test, label="Thực tế (Test)", color="blue", linewidth=2
)
plt.plot(
    us_aqi_test.index,
    sarima_predictions,
    label="Dự báo (SARIMA)",
    color="red",
    linestyle="--",
    linewidth=2,
)

plt.title(
    "So Sánh AQI Thực Tế Và Dự Báo - Mô Hình SARIMA(1, 0, 3)",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Thời gian", fontsize=12)
plt.ylabel("Chỉ số US AQI", fontsize=12)
plt.legend(fontsize=11, loc="best")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "sarima_aqi")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")

In [ ]:
# Dự đoán và đánh giá
y_pred_sarima_aqi = []
conf_int_lower = []
conf_int_upper = []

current_model = copy.deepcopy(sarima_model)
# Dự báo cuốn chiếu (Walk-forward Validation / Rolling Forecast)
for i in range(len(us_aqi_test)):
    # Dự đoán 1 bước tiếp theo
    pred, ci = current_model.predict(n_periods=1, return_conf_int=True, alpha=0.05)

    # pred và ci trả về dưới dạng mảng (array), lấy phần tử đầu tiên [0]
    pred_value = pred.iloc[0] if isinstance(pred, pd.Series) else pred[0]

    # Lưu giá trị dự báo
    y_pred_sarima_aqi.append(np.clip(pred_value, 0, 500))
    conf_int_lower.append(ci[0][0])
    conf_int_upper.append(ci[0][1])

    # "Cho model thấy" giá trị thật của ngày này, rồi tiếp tục dự đoán bước kế (không train lại)
    true_value = us_aqi_test.iloc[i]
    current_model.update([true_value])

y_pred_sarima_aqi = pd.Series(y_pred_sarima_aqi, index=us_aqi_test.index)
conf_int = pd.DataFrame(
    {"lower": conf_int_lower, "upper": conf_int_upper}, index=us_aqi_test.index
)

evaluate("SARIMA (Rolling Forecast)", us_aqi_test, y_pred_sarima_aqi)

#### **Đánh giá các metrics:**

- **MAE = 12.42 & RMSE = 16.66:** Sai số dự báo trung bình dao động khoảng 12.4 đơn vị AQI. Việc RMSE không chênh lệch quá lớn so với MAE cho thấy mô hình hiếm khi gặp phải các sai số cục bộ quá nghiêm trọng.

- **R² = 0.4002:** Dữ liệu quá khứ của chuỗi AQI tự nó có thể giải thích được khoảng 40% sự biến thiên của ngày kế tiếp. Đây là một mức hiệu suất hoàn toàn hợp lý, thậm chí là rất tốt đối với một cấu trúc mô hình đơn biến (univariate) bị giới hạn thông tin. Việc R² thấp hơn rõ rệt so với mô hình hồi quy đa biến là minh chứng rõ ràng nhất cho giá trị thông tin khổng lồ mà các đặc trưng ngoại sinh (thời tiết, PM2.5, PM10...) mang lại, chứ hoàn toàn không phải do SARIMA bị thiết lập sai kỹ thuật.

- **MAPE = 13.17%:** Tỷ lệ sai số phần trăm trung bình cao hơn mô hình hồi quy. Tương tự như hệ số R², mức chênh lệch này là hệ quả tất yếu khi cấu trúc của SARIMA phải "chiến đấu đơn độc" để dự đoán một hiện tượng phức tạp mà không có sự hỗ trợ của các biến môi trường. Điều này một lần nữa khẳng định quyết định ứng dụng các thuật toán Machine Learning của nhóm là hoàn toàn đúng đắn và cần thiết.

In [ ]:
# Forecast vs Thực tế + Khoảng tin cậy (Rolling Forecast)
plt.figure(figsize=(14, 6))

plt.plot(
    us_aqi_train.index[-60:],
    us_aqi_train.iloc[-60:],
    label="Train (60 ngày cuối)",
    color="gray",
    alpha=0.6,
)
plt.plot(us_aqi_test.index, us_aqi_test, label="Thực tế", color="#2C3E50", linewidth=2)
plt.plot(
    us_aqi_test.index,
    y_pred_sarima_aqi,
    label="SARIMA Rolling Forecast",
    color="#E74C3C",
    linestyle="--",
    linewidth=2,
)
plt.fill_between(
    us_aqi_test.index,
    conf_int["lower"],
    conf_int["upper"],
    color="#E74C3C",
    alpha=0.15,
    label="Khoảng tin cậy 95%",
)

plt.xlabel("Thời gian")
plt.ylabel("Chỉ số US AQI")
plt.title(
    "So Sánh AQI Thực Tế Và Dự Báo - Mô Hình SARIMA (Rolling Forecast)",
    fontweight="bold",
)
plt.legend()
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "sarima_aqi_rolling_forecast")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

In [ ]:
# Lưu mô hình
models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "sarima_model.pkl")
joblib.dump(current_model, save_path)
print("Đã lưu sarima_model.pkl")

#### **Kết luận chung về mô hình SARIMA**

- **Ưu điểm:** Khi kết hợp với Rolling Forecast (dự báo cuốn chiếu), SARIMA thể hiện sự ổn định cao, bám sát tốt xu hướng dao động của chuỗi AQI và khắc phục hoàn toàn hiện tượng cộng dồn sai số. Khoảng tin cậy được duy trì ở mức hợp lý.

- **Hạn chế:** Bản chất tuyến tính và đơn biến khiến SARIMA thiếu nhạy bén trước các cú sốc ngoại cảnh (như thay đổi thời tiết, nguồn phát thải đột ngột). Mô hình thường bị trễ nhịp hoặc dự báo an toàn tại các điểm đỉnh/đáy. Điều này lý giải vì sao hiệu suất (R², MAPE) kém hơn rõ rệt so với mô hình đa biến.

- **Đề xuất:** SARIMA hoạt động rất tốt trong vai trò một **mô hình cơ sở (Baseline)** để đối chiếu. Tuy nhiên, để tối ưu hóa độ chính xác trong bài toán dự báo chất lượng không khí thực tế, nên ưu tiên sử dụng các mô hình Học máy đa biến (như LightGBM, XGBoost...) để tận dụng được sức mạnh của các yếu tố khí tượng và nồng độ chất ô nhiễm thành phần.

### **3.6. So sánh tổng hợp**

Gom kết quả từ tất cả các model đã train ở các mục trước để so sánh trực quan trên cùng một bảng và biểu đồ.

**Lưu ý quan trọng:** Ridge, Decision Tree, Random Forest, LightGBM và XGBoost dự đoán target_reg_tomorrow (AQI ngày mai, dùng đầy đủ features đa biến), trong khi SARIMA dự đoán us_aqi (AQI của chính ngày đó trong test, chỉ dùng quá khứ của chuỗi AQI). Đây là 2 bài toán khác nhau về bản chất thời gian dự đoán nên dùng 2 y_true riêng biệt khi tính metrics - phần phân tích phía dưới sẽ giải thích thêm sự khác biệt này khi diễn giải kết quả.

In [ ]:
# Gom dự đoán của tất cả các model
MODEL_PREDICTIONS = {
    "Ridge": y_pred_ridge,
    "Decision Tree": y_pred_dt,
    "Random Forest": y_pred_rf,
    "LightGBM": y_pred_lgbm,
    "XGBoost": y_pred_xgb,
    "SARIMA": y_pred_sarima_aqi,
}

Y_TRUE_ML = y_test_reg  # target_reg_tomorrow - dùng cho Ridge/DT/RF/LightGBM/XGBoost
Y_TRUE_SARIMA = us_aqi_test  # us_aqi - chỉ dùng riêng cho SARIMA

results = []
for model_name, y_pred in MODEL_PREDICTIONS.items():
    y_true_use = Y_TRUE_SARIMA if model_name == "SARIMA" else Y_TRUE_ML

    y_true_np = np.array(y_true_use)
    y_pred_np = np.array(y_pred)

    y_true_use = np.clip(y_true_np, 0, 500)
    y_pred = np.clip(y_pred_np, 0, 500)

    mae = mean_absolute_error(y_true_use, y_pred)
    mse = mean_squared_error(y_true_use, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true_use, y_pred)
    mape = np.mean(np.abs((y_true_use - y_pred) / (y_true_use + 1e-9))) * 100

    results.append(
        {
            "Model": model_name,
            "MAE (Càng thấp càng tốt)": mae,
            "RMSE (Càng thấp càng tốt)": rmse,
            "R2 (Càng cao càng tốt)": r2,
            "MAPE (Càng thấp càng tốt)": mape,
        }
    )

df_compare = (
    pd.DataFrame(results).set_index("Model").sort_values("RMSE (Càng thấp càng tốt)")
)

print("Bảng so sánh tổng hợp các mô hình:")
display(
    df_compare.style.highlight_min(
        subset=[
            "MAE (Càng thấp càng tốt)",
            "RMSE (Càng thấp càng tốt)",
            "MAPE (Càng thấp càng tốt)",
        ],
        color="#c8f7c5",
    ).highlight_max(subset=["R2 (Càng cao càng tốt)"], color="#c8f7c5")
)

# Lưu file kết quả
outputs_dir = os.path.join(ROOT, "outputs")
file_path = os.path.join(outputs_dir, "regression_results.csv")
df_compare.to_csv(file_path, encoding="utf-8-sig")
print(f"\nĐã lưu kết quả thành công")

In [ ]:
# Tổng hợp kết quả CV của tất cả models
all_cv = pd.concat(
    [df_cv_ridge, df_cv_dt, df_cv_rf, df_cv_lgbm, df_cv_xgb], ignore_index=True
)
df_summary_cv = (
    cv_summary(all_cv).sort_values("R2_mean", ascending=False).reset_index(drop=True)
)
display(df_summary_cv)

In [ ]:
# Tổng hợp kết quả CV của tất cả models
all_cv = pd.concat(
    [df_cv_ridge, df_cv_dt, df_cv_rf, df_cv_lgbm, df_cv_xgb], ignore_index=True
)
df_summary_cv = (
    cv_summary(all_cv).sort_values("R2_mean", ascending=False).reset_index(drop=True)
)
display(df_summary_cv)

In [ ]:
# Vẽ biểu đồ so sánh hiệu suất các mô hình
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_to_plot = [
    "MAE (Càng thấp càng tốt)",
    "RMSE (Càng thấp càng tốt)",
    "R2 (Càng cao càng tốt)",
    "MAPE (Càng thấp càng tốt)",
]

colors_default = "#3498DB"
color_best = "#27AE60"

for ax, metric in zip(axes.flatten(), metrics_to_plot):
    if metric == "R2 (Càng cao càng tốt)":
        # R2 càng cao càng tốt
        data_sorted = df_compare[metric].sort_values(ascending=False)
        best_model = data_sorted.idxmax()
    else:
        # MAE, RMSE, MAPE càng thấp càng tốt
        data_sorted = df_compare[metric].sort_values(ascending=True)
        best_model = data_sorted.idxmin()

    bar_colors = [
        color_best if m == best_model else colors_default for m in data_sorted.index
    ]

    bars = ax.barh(data_sorted.index, data_sorted.values, color=bar_colors, alpha=0.85)

    ax.invert_yaxis()

    ax.set_title(metric, fontweight="bold", fontsize=12)
    ax.set_xlabel(metric)
    ax.grid(axis="x", alpha=0.3)

    for bar, val in zip(bars, data_sorted.values):
        text_val = f" {val:.2f}%" if metric == "MAPE" else f" {val:.2f}"
        ax.text(
            bar.get_width(),
            bar.get_y() + bar.get_height() / 2,
            text_val,
            va="center",
            fontsize=11,
        )

plt.suptitle("So Sánh Hiệu Suất Các Mô Hình", fontweight="bold", fontsize=14, y=1.0)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "all_models_compare")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

### **3.7. Đánh giá Best Model**

Phân tích sâu hơn model được chọn là tốt nhất (dựa trên kết quả Mục 3.6) - bao gồm biểu đồ Actual vs Predicted, Scatter, Residuals và phân tích lỗi theo từng tháng.

#### **Biểu đồ Actual vs Predicted theo thời gian**

In [ ]:
def plot_all_models(results):
    fig, axes = plt.subplots(3, 2, figsize=(18, 15))
    axes = axes.flatten()

    for i, (model_name, (y_true, y_pred)) in enumerate(results.items()):
        ax = axes[i]
        ax.plot(
            y_true.index, y_true.values, label="Thực tế", color="#2196F3", alpha=0.6
        )
        ax.plot(y_true.index, y_pred, label="Dự đoán", color="#FF9800", alpha=0.8)

        ax.set_title(f"Model: {model_name}", fontsize=12, fontweight="bold")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    models_dir = os.path.join(figures_dir, "models")
    file_path = os.path.join(models_dir, "all_models_actual_predicted")
    plt.savefig(file_path, bbox_inches="tight")
    print("Đã lưu biểu đồ thành công")
    plt.show()

In [ ]:
results = {
    "Ridge Regression": (y_test_reg, y_pred_ridge),
    "Decision Tree": (y_test_reg, y_pred_dt),
    "Random Forest": (y_test_reg, y_pred_rf),
    "LightGBM": (y_test_reg, y_pred_lgbm),
    "XGBoost": (y_test_reg, y_pred_xgb),
    "SARIMA": (us_aqi_test, y_pred_sarima_aqi),
}
plot_all_models(results)

#### **Scatter Plot + Residuals**

In [ ]:
def plot_all_models_scatter_residuals(results):
    fig, axes = plt.subplots(6, 2, figsize=(16, 24))

    for i, (model_name, (y_true, y_pred)) in enumerate(results.items()):
        # Cột 1: Scatter Plot
        ax_scatter = axes[i, 0]
        ax_scatter.scatter(y_true, y_pred, alpha=0.5, color="#4CAF50")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        ax_scatter.plot(
            [min_val, max_val],
            [min_val, max_val],
            color="red",
            linestyle="--",
            linewidth=2,
        )
        ax_scatter.set_title(f"[{model_name}] Scatter Plot", fontweight="bold")
        ax_scatter.set_xlabel("Thực tế")
        ax_scatter.set_ylabel("Dự đoán")

        # Cột 2: Residuals Histogram
        ax_hist = axes[i, 1]
        residuals = y_true - y_pred
        sns.histplot(residuals, kde=True, ax=ax_hist, color="#9C27B0")
        ax_hist.axvline(0, color="red", linestyle="--", linewidth=2)
        ax_hist.set_title(f"[{model_name}] Residuals", fontweight="bold")
        ax_hist.set_xlabel("Sai số (Thực tế - Dự đoán)")
        ax_hist.set_ylabel("Tần suất")

    plt.tight_layout()
    models_dir = os.path.join(figures_dir, "models")
    file_path = os.path.join(models_dir, "all_models_scatter_residual")
    plt.savefig(file_path, bbox_inches="tight")
    print("Đã lưu biểu đồ thành công")
    plt.show()

In [ ]:
results = {
    "Ridge Regression": (y_test_reg, y_pred_ridge),
    "Decision Tree": (y_test_reg, y_pred_dt),
    "Random Forest": (y_test_reg, y_pred_rf),
    "LightGBM": (y_test_reg, y_pred_lgbm),
    "XGBoost": (y_test_reg, y_pred_xgb),
    "SARIMA": (us_aqi_test, y_pred_sarima_aqi),
}
plot_all_models_scatter_residuals(results)

#### **MAE theo từng tháng - phát hiện tháng nào model sai nhiều nhất**

In [ ]:
def plot_all_models_mae_by_month(results):
    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    axes = axes.flatten()

    summary_list = []

    for i, (model_name, (y_true, y_pred)) in enumerate(results.items()):
        ax = axes[i]

        # 1. Tính toán MAE
        df_eval = pd.DataFrame({"Actual": y_true, "Predicted": y_pred})
        df_eval["Month"] = df_eval.index.month
        df_eval["Abs_Error"] = np.abs(df_eval["Actual"] - df_eval["Predicted"])
        mae_by_month = df_eval.groupby("Month")["Abs_Error"].mean()

        # 2. Vẽ biểu đồ
        sns.barplot(
            x=mae_by_month.index, y=mae_by_month.values, palette="viridis", ax=ax
        )

        ax.set_title(f"[{model_name}] MAE Theo Tháng", fontsize=12, fontweight="bold")
        ax.set_xlabel("Tháng")
        ax.set_ylabel("MAE")
        ax.grid(axis="y", linestyle="--", alpha=0.5)

        # 3. Lưu kết quả tháng tệ nhất
        worst_month = mae_by_month.idxmax()
        worst_mae = mae_by_month.max()
        summary_list.append(
            {"Model": model_name, "Worst_Month": int(worst_month), "Max_MAE": worst_mae}
        )

    plt.tight_layout()
    models_dir = os.path.join(figures_dir, "models")
    file_path = os.path.join(models_dir, "all_models_mae_by_month")
    plt.savefig(file_path, bbox_inches="tight")
    print("Đã lưu biểu đồ thành công")
    plt.show()

    # In bảng tổng kết
    print("TỔNG KẾT THÁNG CÓ SAI SỐ CAO NHẤT")
    summary_df = pd.DataFrame(summary_list)
    print(summary_df.to_string(index=False))

In [ ]:
results = {
    "Ridge Regression": (y_test_reg, y_pred_ridge),
    "Decision Tree": (y_test_reg, y_pred_dt),
    "Random Forest": (y_test_reg, y_pred_rf),
    "LightGBM": (y_test_reg, y_pred_lgbm),
    "XGBoost": (y_test_reg, y_pred_xgb),
    "SARIMA": (us_aqi_test, y_pred_sarima_aqi),
}
plot_all_models_mae_by_month(results)

#### **Kết luận đánh giá Best Model:**

Dựa trên sự nhất quán từ các biểu đồ trực quan, **Ridge Regression** hoàn toàn xứng đáng là mô hình tốt nhất (Best Model) cho bài toán dự báo này với các lập luận sau:

1. **Tính ổn định và Khả năng tổng quát hóa cao:** Nhờ cơ chế L2 Regularization, Ridge đã xử lý tốt nhiễu dữ liệu, mang lại một đường dự đoán bám sát xu hướng thực tế mà không bị nhạy cảm thái quá với các điểm bất thường (outliers).
2. **Độ tin cậy của thuật toán:** Biểu đồ phần dư hội tụ tại 0 chứng minh Ridge là một mô hình không thiên kiến (unbiased). Sai số của mô hình mang tính ngẫu nhiên nhiều hơn là lỗi do cấu trúc của thuật toán.
3. **Sự đánh đổi hợp lý:** Việc mô hình dự đoán thấp hơn một chút ở các đỉnh ô nhiễm cực đoan là một sự đánh đổi (bias-variance tradeoff) có thể chấp nhận được ở các mô hình tuyến tính, nhằm đổi lấy sự chính xác và sai số rất thấp ở phần lớn các ngày còn lại trong năm.

Tổng thể, mô hình Ridge cung cấp một kết quả dự báo vừa đủ độ phức tạp để nắm bắt quy luật, vừa đủ sự tinh gọn để duy trì tính chính xác cao và ổn định qua thời gian.

### **3.8. Phân tích SARIMA với Machine Learning**

In [ ]:
# Trực quan hóa SARIMA vs từng ML model

# Mỗi model 1 subplot riêng, cùng trục y (AQI) để so sánh độ khít giữa các
# model. Vùng tô màu giữa đường dự đoán và đường thực tế giúp nhìn trực quan
# sai số lớn/nhỏ tại từng thời điểm.

n_models = len(MODEL_PREDICTIONS)
ncols = 2
nrows = (n_models + 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4.5 * nrows), sharey=True)
axes = axes.flatten()

model_colors = {
    "Ridge": "#9B59B6",
    "Decision Tree": "#F39C12",
    "Random Forest": "#3498DB",
    "LightGBM": "#27AE60",
    "XGBoost": "#E67E22",
    "SARIMA": "#E74C3C",
}

for ax, (model_name, y_pred) in zip(axes, MODEL_PREDICTIONS.items()):
    is_sarima = model_name == "SARIMA"
    y_true_use = Y_TRUE_SARIMA if is_sarima else Y_TRUE_ML
    x_index = y_true_use.index

    y_true_use = np.clip(np.array(y_true_use), 0, 500)
    y_pred = np.clip(np.array(y_pred), 0, 500)

    # Vẽ đường thực tế
    ax.plot(x_index, y_true_use, label="Thực tế", color="#2C3E50", linewidth=1.8)
    # Vẽ đường dự đoán
    ax.plot(
        x_index,
        y_pred,
        label=model_name,
        color=model_colors[model_name],
        linestyle="--",
        linewidth=1.5,
        alpha=0.9,
    )

    ax.fill_between(
        x_index, y_true_use, y_pred, color=model_colors[model_name], alpha=0.12
    )

    # Lấy lại metrics đã tính ở df_results để hiện ngay trên subplot
    rmse_val = df_compare.loc[model_name, "RMSE (Càng thấp càng tốt)"]
    r2_val = df_compare.loc[model_name, "R2 (Càng cao càng tốt)"]
    subtitle = f"{model_name} (RMSE={rmse_val:.2f}, R²={r2_val:.2f})"
    if is_sarima:
        subtitle += "\n[dự đoán us_aqi, không phải target_reg_tomorrow]"

    ax.set_title(subtitle, fontweight="bold", fontsize=11)
    ax.set_ylabel("Chỉ số US AQI")
    ax.legend(loc="upper right", fontsize=9)
    ax.tick_params(axis="x", rotation=20)

# Ẩn subplot dư (nếu số model lẻ, ô cuối lưới sẽ trống)
for ax in axes[n_models:]:
    ax.set_visible(False)

plt.suptitle(
    "So Sánh Chi Tiết: SARIMA vs Từng Model ML", fontweight="bold", fontsize=14, y=1.0
)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "sarima_compare")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Đánh giá giới hạn của ARIMA và lý do đề xuất Machine Learning:**

- Mặc dù cấu trúc ARIMA(1, 0, 3) nắm bắt rất tốt các quy luật tự tương quan tuyến tính trong ngắn hạn (thể hiện qua việc triệt tiêu được nhiễu trong tập Train), nhưng khi thực hiện dự báo dài hạn trên tập Test, mô hình nhanh chóng gặp hiện tượng hội tụ về giá trị trung bình (Mean Reversion). Điều này tạo ra một đường dự báo đi ngang, hoàn toàn "bắt trượt" các biến động đỉnh nhọn và xu hướng phức tạp của chỉ số AQI thực tế.

- **Từ kết quả này, nhóm rút ra kết luận:** Các mô hình thống kê tuyến tính truyền thống không đủ sức chứa để xử lý tính phi tuyến và độ nhiễu cao của dữ liệu môi trường nếu không có sự hỗ trợ của các biến ngoại sinh. Do đó, kết quả dự báo và các chỉ số sai số (MAE, RMSE) của ARIMA sẽ được nhóm sử dụng làm Baseline. Mục đích là để làm thước đo so sánh, qua đó làm nổi bật tính ưu việt, độ linh hoạt và khả năng dự báo bám sát thực tế của các thuật toán Machine Learning ở phần tiếp theo của dự án.

- Điều này cũng khẳng định rằng nhóm đã thực sự làm tốt trong việc **Feature Engineering**. Cũng như những giá trị mà việc feature engineering mang lại rất lớn chứ không phải chỉ là bước làm cho có.

### **3.9. Tổng kết**

#### **Best Model được chọn: Ridge Regression**

- **Hiệu suất:**
    - Mô hình đạt độ phù hợp và khả năng giải thích cao nhất với
      $R^2 \approx 0.75$, cho thấy khả năng nắm bắt được 75% sự biến
      thiên của chỉ số AQI thực tế.
    - Các chỉ số đo lường sai số đều đạt mức tối ưu nhất:
      $\text{MAE} \approx 8.27$ và $\text{RMSE} \approx 10.78$.
    - Đặc biệt, chỉ số sai số phần trăm tuyệt đối trung bình
      $\text{MAPE} \approx 8.86\%$. Mức sai số dưới 10% khẳng định
      độ lệch tương đối của mô hình là rất nhỏ, hoàn toàn đáp ứng
      tốt độ tin cậy cho việc dự báo chất lượng không khí trong
      thực tế.

- **Độ ổn định cao qua kiểm định chéo (Cross-Validation):**
    - Khi tiến hành kiểm định chéo để đánh giá tính bền vững, mô
      hình duy trì hiệu năng cao với trung bình $\text{R2\_mean} =
      0.70$, đi kèm mức sai số thấp ($\text{MAE\_mean} = 7.73$,
      $\text{RMSE\_mean} = 10.24$).
    - Độ dao động giữa các lần kiểm định ($\text{R2\_std} = 0.10$)
      nằm trong ngưỡng an toàn, minh chứng cho việc mô hình không
      bị quá khớp (overfitting) và có khả năng tổng quát hóa tốt
      trên những tập dữ liệu chưa từng thấy.

- **Độ tương thích với bản chất dữ liệu & Tính tối ưu tính toán:**
    - Việc thuật toán tuyến tính đạt kết quả cao cho thấy mối quan
      hệ giữa các biến đầu vào và biến mục tiêu (AQI) mang tính
      tuyến tính rõ rệt - điều này được lý giải bởi bước Feature
      Engineering kỹ lưỡng của nhóm: Các đặc trưng lag,
      rolling, sin/cos đã "tuyến tính hóa" phần lớn tính phi tuyến
      vốn có của chuỗi thời gian AQI.
    - Cơ chế điều chuẩn $L_2$ (L2 Regularization) tích hợp trong
      mô hình đã triệt tiêu hiệu quả hiện tượng đa cộng tuyến
      (multicollinearity) - một vấn đề phổ biến trong dữ liệu chuỗi
      thời gian khi các features lag/rolling có tương quan cao với
      nhau (ví dụ: `t-1` và `rolling_mean_3` có r = 0.94).
    - Mô hình gọn nhẹ, thời gian huấn luyện nhanh và dễ diễn giải
      kết quả thông qua hệ số hồi quy (coefficients) - feature nào
      có hệ số lớn đóng góp nhiều vào dự đoán, không cần công cụ
      phức tạp như SHAP để giải thích cơ bản.
    - Kết quả này minh chứng cho nguyên lý **"Occam's Razor"** trong
      Machine Learning: Mô hình đơn giản hơn không nhất thiết kém
      hơn, đặc biệt khi dữ liệu đã được Feature Engineering tốt và
      kích thước mẫu có hạn (~1.013 mẫu Train).

---

#### **Giới hạn của mô hình:**

- **Thiếu dữ liệu thời tiết:** Dataset hiện tại chỉ bao gồm các chỉ
  số ô nhiễm (PM2.5, PM10, ozone...) mà không có các biến khí tượng
  quan trọng như nhiệt độ, độ ẩm, tốc độ gió, lượng mưa - những
  yếu tố ảnh hưởng trực tiếp đến khả năng phân tán và tích tụ bụi.
  Đây là nguyên nhân chính khiến mô hình khó dự đoán chính xác ở
  các ngày chuyển mùa (tháng 4, tháng 6, tháng 10).

- **Hạn chề về số lượng trạm quan trắc:** Chỉ có 5 trạm quan trắc - không đại diện được cho toàn bộ TP. Hồ Chí Minh vốn có diện tích lớn và mức độ ô nhiễm không đồng đều giữa các khu vực (trung tâm, ngoại ô, khu công nghiệp). Và quan trọng hơn, khả năng cao dữ liệu chỉ được lấy từ 1 trạm quan trắc nên càng nghiệm trọng hơn.

- **Kích thước dataset hạn chế:** Với ~1.298 ngày (khoảng 3.5 năm),
  mô hình chưa quan sát đủ nhiều chu kỳ mùa vụ để học pattern dài
  hạn ổn định. Các mô hình phức tạp hơn (LSTM, Transformer) thường
  cần ít nhất 5-10 năm dữ liệu để phát huy ưu thế.

- **Mô hình không tự cập nhật:** `best_regressor.pkl` được train cố định
  tại một thời điểm. Nếu xu hướng AQI thay đổi đáng kể (ví dụ do
  chính sách kiểm soát khí thải mới), mô hình sẽ dần kém chính xác
  theo thời gian mà không tự nhận ra.

- **Ridge không nắm bắt được phi tuyến cục bộ:** Dù Feature
  Engineering đã giảm phần lớn tính phi tuyến, các đợt ô nhiễm đột
  biến ngắn hạn (spike trong 1-2 ngày, thường do cháy rừng lân cận
  hoặc thời tiết cực đoan) vẫn khiến Ridge dự đoán sai lệch đáng
  kể - đây là loại sự kiện mà tree-based models có thể xử lý tốt
  hơn nếu có đủ dữ liệu.

---

#### **Hướng phát triển tiếp theo:**

- **Bổ sung dữ liệu thời tiết:** Tích hợp thêm nhiệt độ, độ ẩm,
  tốc độ gió, lượng mưa từ OpenWeatherMap API hoặc ERA5 - kỳ vọng
  cải thiện R² thêm 0.05-0.10, đặc biệt ở các tháng chuyển mùa.

- **Mở rộng dữ liệu nhiều trạm:** Thu thập thêm dữ liệu từ các
  trạm quan trắc khác ở TP. Hồ Chí Minh để mô hình học được sự
  phân bố ô nhiễm không gian, không chỉ theo thời gian.

- **Thử nghiệm LSTM/Transformer:** Khi có thêm dữ liệu (>5 năm),
  các mô hình deep learning cho chuỗi thời gian (LSTM, Temporal
  Fusion Transformer) có thể khai thác pattern phức tạp mà Ridge
  không nắm bắt được.

- **Tự động cập nhật model:** Xây dựng pipeline tái huấn luyện định
  kỳ (ví dụ mỗi 3-6 tháng) khi tích lũy đủ dữ liệu mới - giúp mô
  hình thích nghi với sự thay đổi xu hướng dài hạn của AQI TP. Hồ
  Chí Minh.

- **Triển khai hệ thống cảnh báo tự động:** Tích hợp dự đoán vào
  hệ thống gửi thông báo tự động (email, SMS, push notification) khi
  AQI dự đoán vượt ngưỡng 100 - chuyển từ "công cụ phân tích" thành
  "hệ thống hỗ trợ quyết định" thực sự cho người dân và cơ quan
  quản lý môi trường.

In [ ]:
# Lưu best model chính thức + metadata.json để Dashboard và SHAP sử dụng
best_model = ridge_reg

models_dir = os.path.join(ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "best_regressor.pkl")
joblib.dump(best_model, save_path)

metadata = {
    "best_model": "Rigde Regression",
    "target_col": "target_reg_tomorrow",
    "n_features": X_train.shape[1],
    "train_period": {
        "start": str(X_train.index.min().date()),
        "end": str(X_train.index.max().date()),
    },
    "test_metrics": {
        "MAE": 8.27,
        "RMSE": 10.78,
        "R²": 0.75,
        "MAPE": 8.86,
    },
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

with open(os.path.join(models_dir, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Đã lưu best_regressor.pkl và metadata.json")


## **04. Model Classification**

#### **Mục tiêu:**

- Xây dựng và huấn luyện các mô hình Machine Learning để dự đoán mức độ chất lượng không khí ngày hôm sau (bài toán Classification) - bổ sung cho bài toán Regression.
- So sánh hiệu suất giữa các mô hình phân loại: Random Forest Classifier, XGBoost Classifier và LightGBM Classifier.
- Phát hiện và xử lý vấn đề mất cân bằng dữ liệu - phần lớn các ngày rơi vào mức "Moderate", trong khi các mức "Good" và "Unhealthy" chiếm tỷ lệ rất nhỏ
- Đánh giá mô hình bằng các chỉ số phù hợp với bài toán phân loại đa lớp: Accuracy, Precision, Recall, F1-score, Confusion Matrix.
- Xây dựng bảng khuyến nghị hành động cụ thể tương ứng với từng mức AQI dự đoán
- Lựa chọn Best Classifier và lưu lại (`best_classifier.pkl`) để sử dụng trong `predict.py` và Dashboard.
- Tạo cơ chế "2 mô hình đồng thuận": Kết hợp kết quả Classification với kết quả Regression để tăng độ tin cậy khi dự đoán thực tế.

#### **Cấu trúc:**
1. Phân tích mất cân bằng
2. Xây dựng mô hình
3. Đánh giá và so sánh
4. Confusion Matrix
5. Hành động thực tiễn
6. Tổng kết

In [ ]:
y_train_cls = y_train["target_class_tomorrow"]
y_test_cls = y_test["target_class_tomorrow"]

### **4.1. Phân tích mất cân bằng**

In [ ]:
# Kiểm tra class balance
df = pd.concat([X_train, X_test], axis=0)
df["target"] = pd.concat([y_train_cls, y_test_cls], axis=0)

df["target"].value_counts(normalize=True).sort_index().plot(
    kind="bar", xlabel="Pollution Level", ylabel="Frequency", title="Class Balance"
)
plt.xticks(rotation=0)
plt.grid(False)
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "imbalanced_data")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Phân tích hiện trạng mất cân bằng dữ liệu (Imbalanced Data)**

- **Thực trạng từ biểu đồ Class Balance:** Tập dữ liệu đang đối mặt với tình trạng mất cân bằng phân lớp cực kỳ nghiêm trọng, bao gồm 4 nhãn (0, 1, 2, 3). Trong đó:

Nhãn **1** chiếm tỷ lệ áp đảo tuyệt đối lên tới xấp xỉ **76.5%**.

Nhãn **2** chiếm khoảng **18%**.

Các nhãn **0** và **3** là những trường hợp đặc biệt, lần lượt chỉ chiếm khoảng **4%** và **1%**.

- **Lý do thực tế tại TP.HCM:**
Đặc thù khí hậu ổn định kết hợp với lượng khí thải giao thông/công nghiệp duy trì đều đặn khiến phần lớn các ngày trong năm neo ở một ngưỡng chất lượng không khí nhất định (Nhãn 1). Những ngày có không khí cực kỳ trong lành (sau mưa bão) hoặc ô nhiễm nghiêm trọng hiếm khi xảy ra.

- **Lựa chọn chiến lược xử lý:**
Với phân bố có lớp thiểu số chỉ chiếm 1%, việc sử dụng các kỹ thuật tái lấy mẫu (Resampling) như Undersampling sẽ làm mất mát gần như toàn bộ dữ liệu, trong khi SMOTE (Oversampling) dễ sinh ra nhiễu do thiếu điểm gốc.

> Do đó, giải pháp tối ưu và an toàn nhất là sử dụng **Trọng số phạt (Cost-sensitive learning)** thông qua tham số `class_weight='balanced'`. Cách này sẽ không thay đổi dữ liệu gốc, mà trực tiếp điều chỉnh thuật toán: phạt rất nặng khi mô hình đoán sai các lớp hiếm (0, 3) và châm chước hơn với lớp đa số (1), từ đó kéo lại sự công bằng trong hàm mất mát.

### **4.2. Xây dựng mô hình**

#### **Random Forest Classifier**

In [ ]:
# Khởi tạo mô hình
rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

# Train mô hình
rf_clf.fit(X_train, y_train_cls)

In [ ]:
# Dự đoán
y_pred_rf = rf_clf.predict(X_test)

# In báo cáo kết quả chi tiết
print("\n---------- KẾT QUẢ RANDOM FOREST ----------")
# Phân tích theo từng lớp 0, 1, 2, 3
print(classification_report(y_test_cls, y_pred_rf, labels=[0, 1, 2, 3]))

In [ ]:
# # 1. Định nghĩa mô hình gốc
# rf_base = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# # 2. Grid để dò
# param_grid = {
#     "n_estimators": [200, 300, 500],
#     "max_depth": [10, 15, 20, None],
#     "min_samples_split": [2, 5, 10],
#     "min_samples_leaf": [1, 2, 4],
# }

# # 3. Vì chia tới 5 folds mà dữ liệu quá mất cân bằng nên thêm stratify vào cho các class đều ở 5 fold
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# # 4. Grid Search
# grid_search = GridSearchCV(
#     estimator=rf_base,
#     param_grid=param_grid,
#     cv=skf,
#     scoring="f1_macro",
#     verbose=1,
#     n_jobs=-1,
# )

# # 5. Fit
# grid_search.fit(X_train, y_train_cls)

**Lưu ý:** Sẽ không chạy lại phần GridSearch cho Random Forest vì rất rất tốn thời gian (Từ 30p - 1h30). Nhóm đã chạy sẵn và chọn ra được bộ tham số ổn định, tối ưu cho mô hình này bao gồm:
- n_estimators = 300
- max_depth = 10
- min_samples_leaf = 4
- min_samples_split = 2

In [ ]:
# Khởi tạo lại mô hình với bộ tham số tối ưu
best_params = {
    "n_estimators": 300,
    "max_depth": 10,
    "class_weight": "balanced",
    "min_samples_split": 2,
    "min_samples_leaf": 4,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": 0,
}

best_rf = RandomForestClassifier(**best_params)

# Train mô hình
best_rf.fit(X_train, y_train_cls)

In [ ]:
y_pred_best_rf = best_rf.predict(X_test)

print("\n---------- KẾT QUẢ RANDOM FOREST SAU KHI GRID SEARCH ----------")
print(classification_report(y_test_cls, y_pred_best_rf, labels=[0, 1, 2, 3]))

**Kết luận sau quá trình Tune Hyperparameter:**

- **Nguyên lý đánh đổi (Trade-off):** Lưới Grid Search đã lựa chọn một kiến trúc cây phòng thủ (`max_depth=10`, `min_samples_leaf=4`) để chống Overfitting. Thuật toán từ chối việc chia nhánh quá sâu để "học vẹt" 5 mẫu Lớp 3 dị biệt.

- **Giới hạn của Random Forest:** Dù đã áp dụng `class_weight='balanced'`, thuật toán Bagging vẫn cho thấy sự đuối sức trong việc đẩy ranh giới quyết định vươn tới các cực trị hiếm gặp của dữ liệu chuỗi thời gian phân bố bất đối xứng.


#### **XGBoost Classifier**

**Chiến lược xử lý Imbalanced Data với XGBoost:**

Khác với Random Forest, thuật toán XGBoost không hỗ trợ trực tiếp tham số `class_weight` cho bài toán phân loại đa lớp (Multi-class classification). Thay vào đó, nhóm sử dụng hàm `compute_sample_weight` của scikit-learn để tính toán trọng số trừng phạt cho từng điểm dữ liệu riêng lẻ trong tập Train.

Những ngày thuộc Lớp 3 (Unhealthy - cực hiếm) sẽ được gán một trọng số cực lớn. Khi mô hình XGBoost xây dựng các cây nối tiếp nhau (Boosting), nó sẽ bị ép phải "nhìn thấy" và tập trung sửa sai cho những mẫu có trọng số khổng lồ này.

In [ ]:
# 1. Tính trọng số cho từng dòng
# Lớp nào càng ít, dòng dữ liệu đó sẽ có trọng số càng cao
weights = compute_sample_weight(class_weight="balanced", y=y_train_cls)

# 2. Khởi tạo mô hình
xgb_clf = XGBClassifier(
    objective="multi:softmax",
    num_class=4,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
)

# 3. Fit model (Ép mô hình học theo trọng số)
xgb_clf.fit(X_train, y_train_cls, sample_weight=weights)

In [ ]:
# Dự đoán
y_pred_xgb = xgb_clf.predict(X_test)

# In báo cáo kết quả chi tiết
print("\n---------- KẾT QUẢ XGBOOST ----------")
print(classification_report(y_test_cls, y_pred_rf, labels=[0, 1, 2, 3]))

**Đánh giá mô hình XGBoost và Ràng buộc Thuật toán (Algorithmic Constraints):**

- Dựa trên báo cáo phân lớp, mô hình XGBoost Classifier đã bộc lộ những giới hạn rõ rệt khi đối mặt với dữ liệu mất cân bằng cực đoan (lớp 3 và lớp 0 chỉ chiếm rất rất nhỏ):

1. **Hiệu ứng triệt tiêu lớp thiểu số (Recall Lớp 3 = 0.00):** Mặc dù nhóm đã thiết lập mảng trọng số `sample_weight` nhằm ép thuật toán chú ý đến Lớp 3, XGBoost vẫn bỏ qua toàn bộ 5 ngày ô nhiễm này. Nguyên nhân cốt lõi nằm ở cơ chế cắt tỉa cành (Pruning) và kiểm soát nhiễu cực kỳ khắt khe của XGBoost. Thuật toán đã đánh giá nhóm 1% dữ liệu mang trọng số bất thường này là outliers và gạt bỏ để bảo vệ tính tổng quát của toàn bộ cây.

2. **Sự sụt giảm hiệu năng so với Random Forest:** Việc cố gắng học nối tiếp để tối ưu hóa hàm mất mát đã gây ra tác dụng phụ lên Lớp 2. Chỉ số Recall của Lớp 2 sụt giảm nghiêm trọng từ 67% (ở Random Forest) xuống chỉ còn 40%.

- **Giải thích về việc không tinh chỉnh siêu tham số (No Hyperparameter Tuning):**
Nhóm quyết định không thực hiện Grid Search cho XGBoost. Lý do là để ép XGBoost chú ý đến Lớp 3, nên buộc phải vô hiệu hóa các cơ chế phòng thủ gốc của nó (như hạ `min_child_weight` hay `gamma` về 0). Thao tác này sẽ đẩy mô hình vào trạng thái Overfitting mất kiểm soát và phá vỡ ranh giới của các lớp đa số, sự đánh đổi này là hoàn toàn không xứng đáng.

- **Hướng giải quyết:**
Chuyển trọng tâm sang thuật toán **LightGBM Classifier**. Với cơ chế mọc cây theo chiều lá (Leaf-wise growth) có khả năng cô lập các nhóm dữ liệu nhỏ rất nhạy bén, kết hợp với việc hỗ trợ tham số `class_weight='balanced'` trực tiếp từ trong lõi thuật toán đa lớp (điều mà XGBoost thiếu sót), LightGBM được kỳ vọng là điểm rơi tối ưu nhất cho bộ dữ liệu này.

#### **LightGBM Classifier**

In [ ]:
# Khởi tạo mô hình
lgbm_clf = lgb.LGBMClassifier(
    class_weight="balanced",
    n_estimators=300,
    max_depth=10,
    learning_rate=0.05,  # Học chậm lại để không bỏ sót Lớp 3
    random_state=42,
    n_jobs=-1,
    verbose=-1,  # Không hiển thị các log
)

# Train mô hình
lgbm_clf.fit(X_train, y_train_cls)

In [ ]:
# Dự đoán
y_pred_lgbm = lgbm_clf.predict(X_test)

print("\n---------- KẾT QUẢ LIGHTGBM ----------")
print(classification_report(y_test_cls, y_pred_lgbm))

In [ ]:
# # Khởi tạo mô hình
# lgbm_base = lgb.LGBMClassifier(
#     class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1
# )

# # Không gian tham số ccho mọc cây theo lá (Leaf-wise)
# param_grid_lgbm = {
#     "n_estimators": [200, 300],
#     "learning_rate": [0.01, 0.05],
#     "max_depth": [8, 10, -1],  # -1 nghĩa là không giới hạn độ sâu
#     "num_leaves": [15, 31, 50],  # Giới hạn số lá để tránh Overfitting
#     "min_child_samples": [2, 5, 10],  # Cho phép tạo lá nhỏ để "hứng" Lớp 3
# }

# # Cấu hình Grid Search
# skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# grid_search_lgbm = GridSearchCV(
#     estimator=lgbm_base,
#     param_grid=param_grid_lgbm,
#     cv=skf,
#     scoring="f1_macro",
#     verbose=1,
#     n_jobs=-1,
# )

# # Fit
# grid_search_lgbm.fit(X_train, y_train_cls)

**Lưu ý:** Sẽ không chạy lại phần GridSearch cho LightGBM vì rất rất tốn thời gian (Từ 30p - 1h30). Nhóm đã chạy sẵn và chọn ra được bộ tham số ổn định, tối ưu cho mô hình này bao gồm:
- learning_rate = 0.01
- max_depth = -1 (Không giới hạn)
- min_child_samples = 10
- n_estimators = 300
- num_leaves = 15

In [ ]:
# Khởi tạo lại mô hình với bộ tham số tối ưu
best_params = {
    "n_estimators": 300,
    "max_depth": -1,
    "min_child_samples": 10,
    "num_leaves": 15,
    "learning_rate": 0.01,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
    "class_weight": "balanced",
}

best_lgbm = lgb.LGBMClassifier(**best_params)

# Train mô hình
best_lgbm.fit(X_train, y_train_cls)

In [ ]:
y_pred_best_lgbm = best_lgbm.predict(X_test)

print("\n---------- KẾT QUẢ LIGHTGBM SAU KHI GRID SEARCH ----------")
print(classification_report(y_test_cls, y_pred_best_lgbm, labels=[0, 1, 2, 3]))

**Đánh giá mô hình LightGBM sau tinh chỉnh và Xác định mô hình tối ưu:**

- **Phá vỡ giới hạn dữ liệu dị biệt (Anomaly Detection):**
Việc mở giới hạn cấu trúc lá (`max_depth=-1`, `min_child_samples=10`) kết hợp với tốc độ học chậm (`learning_rate=0.01`) đã tạo ra một bước đột phá. LightGBM đã vượt qua giới hạn Concept Drift, thành công kéo điểm Recall của Lớp 3 (Unhealthy) từ 0.00 lên 0.60. Điều này chứng minh thuật toán đã thực sự bóc tách được các ranh giới đặc trưng cực nhỏ của những ngày ô nhiễm nghiêm trọng.

- **Sự ổn định và Khả năng tổng quát hóa của mô hình:**
Việc Accuracy tổng thể không thay đổi là một điểm sáng cho việc Grid Search. Chỉ số đại diện cho sự cân bằng tổng thể là Macro F1 đã tăng từ 0.42 lên 0.5, xác nhận mô hình không còn bị thiên vị (bias) mà đã học được bức tranh toàn cảnh khách quan hơn.

> **Kết luận:** Với khả năng nhận diện được cả rủi ro cao nhất (Lớp 3), `LightGBM Classifier (Tuned)` được lựa chọn làm mô hình tốt nhất của dự án để triển khai các bước tiếp theo.

### **4.3. Đánh giá và so sánh**

#### **So sánh 3 mô hình**

In [ ]:
# Gom các dự đoán vào một từ điển
model_preds = {
    "Random Forest (Tuned)": y_pred_best_rf,
    "XGBoost (Based)": y_pred_xgb,
    "LightGBM (Tuned)": y_pred_best_lgbm,
}
labels = [0, 1, 2, 3]

# Tính các chỉ số
comparison_data = []
for model_name, y_pred in model_preds.items():
    acc = accuracy_score(y_test_cls, y_pred)
    precision_macro = precision_score(
        y_test_cls, y_pred, labels=labels, average="macro", zero_division=0
    )
    recall_macro = recall_score(
        y_test_cls, y_pred, labels=labels, average="macro", zero_division=0
    )
    f1_macro = f1_score(
        y_test_cls, y_pred, labels=labels, average="macro", zero_division=0
    )
    f1_weight = f1_score(
        y_test_cls, y_pred, labels=labels, average="weighted", zero_division=0
    )

    comparison_data.append(
        {
            "Mô hình": model_name,
            "Accuracy": round(acc, 4),
            "Precision (Macro)": round(precision_macro, 4),
            "Recall (Macro)": round(recall_macro, 4),
            "F1-Score (Macro)": round(f1_macro, 4),
            "F1-Score (Weighted)": round(f1_weight, 4),
        }
    )

# Hiển thị
df_compare = pd.DataFrame(comparison_data).set_index("Mô hình")
print("---------- BẢNG SO SÁNH HIỆU NĂNG 3 MÔ HÌNH CLASSIFICATION ----------")
display(df_compare)

# Lưu file kết quả
outputs_dir = os.path.join(ROOT, "outputs")
file_path = os.path.join(outputs_dir, "classification_results.csv")
df_compare.to_csv(file_path, encoding="utf-8-sig")
print(f"\nĐã lưu kết quả thành công")

#### **Trực quan so sánh 3 mô hình**

In [ ]:
df_plot = df_compare.T

ax = df_plot.plot(
    kind="bar", figsize=(14, 7), colormap="Set2", edgecolor="black", width=0.7
)

plt.title("So Sánh Các Mô Hình Theo Metrics", fontweight="bold", fontsize=14, pad=15)
plt.xlabel("Metrics", fontweight="bold", fontsize=11)
plt.ylabel("Score", fontweight="bold", fontsize=11)

plt.xticks(rotation=0, fontweight="bold", fontsize=11)

plt.legend(title="Mô Hình", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=11)

# In giá trị lên đầu mỗi bar
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(
            f"{height:.2f}",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="bottom",
            fontsize=10,
            color="black",
            fontweight="bold",
            xytext=(0, 4),
            textcoords="offset points",
        )

plt.ylim(0, 1.15)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.grid(False)
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "all_model_classifier_compare")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

### **4.4. Confusion Matrix**

In [ ]:
display_labels = ["Good (0)", "Moderate (1)", "Sensitive (2)", "Unhealthy (3)"]

fig, ax = plt.subplots()

ConfusionMatrixDisplay.from_predictions(
    y_test_cls,
    y_pred_best_lgbm,
    display_labels=display_labels,
    cmap="Oranges",
    colorbar=False,
    ax=ax,
)

ax.set_title(
    "Confusion Matrix - Best Classifier (Tuned LightGBM)",
    fontweight="bold",
    fontsize=14,
    pad=15,
)
ax.set_xlabel("Predicted", fontweight="bold")
ax.set_ylabel("Actual", fontweight="bold")
ax.grid(False)

# Vẽ đường kẻ chia ô
n = len(display_labels)
ax.set_xticks([x - 0.5 for x in range(1, n)], minor=True)
ax.set_yticks([y - 0.5 for y in range(1, n)], minor=True)
ax.grid(which="minor", color="black", linewidth=0.5)

plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "confusion_matrix")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

### **4.5. Hành động thực tiễn**

Mô hình phân loại chất lượng không khí không chỉ dừng lại ở các con số chính xác trên tập Test, mà cấu trúc đầu ra của mô hình (Lớp 0, 1, 2, 3) được ánh xạ trực tiếp thành các khuyến nghị hành động cụ thể trong thực tế đời sống và quản lý đô thị tại TP.HCM:

**Mức Tốt (Good - Lớp 0):**

- Ý nghĩa: Chất lượng không khí lý tưởng, không gây rủi ro cho sức khỏe.

- Hành động:
    - Khuyến khích người dân tích cực tham gia các hoạt động thể thao, giải trí ngoài trời.
    - Các trường học ưu tiên tổ chức các buổi dã ngoại, thể dục ngoại khóa cho học sinh.
    - Hệ thống thông gió của các tòa nhà cao tầng tăng cường lấy khí tươi tự nhiên từ bên ngoài để tiết kiệm điện năng.

**Mức Trung bình (Moderate - Lớp 1):**

- Ý nghĩa: Chất lượng không khí ở mức chấp nhận được nhưng bắt đầu xuất hiện rủi ro nhỏ đối với nhóm người nhạy cảm.

- Hành động:
    - Người dân nói chung vẫn có thể hoạt động ngoài trời bình thường.
    - Nhóm nhạy cảm (trẻ em, người già, người có bệnh nền hen suyễn, tim mạch) nên giảm các hoạt động gắng sức hoặc kéo dài ngoài trời; cân nhắc đóng bớt cửa sổ nếu vị trí nhà ở gần các trục đường giao thông lớn.

**Mức Kém / Nguy hiểm (Unhealthy - Lớp 2 & 3):**

- Ý nghĩa: Không khí chuyển sang ngưỡng có hại, gây ảnh hưởng trực tiếp đến hệ hô hấp của toàn bộ cộng đồng.

- Hành động ứng phó:
    - Đối với người dân: Yêu cầu bắt buộc đeo khẩu trang đạt chuẩn chống bụi mịn (N95, PM2.5) khi di chuyển ngoài đường. Các hộ gia đình đóng kín cửa sổ, bật máy lọc không khí ở chế độ công suất cao. Hạn chế tối đa các hoạt động thể dục buổi sáng ngoài trời.
    - Đối với quản lý đô thị: Kích hoạt ngay hệ thống xe phun nước dập bụi tại các đại lộ lớn; điều tiết phân luồng giao thông từ xa để giảm thiểu hiện tượng kẹt xe giờ cao điểm; tạm dừng các hoạt động thi công đào đường hoặc xây dựng lộ thiên phát tán nhiều bụi cho đến khi chỉ số AQI hạ nhiệt.

### **4.6. Tổng kết**

#### **Lý do lựa chọn LightGBM Classifier (Tuned) làm mô hình tối ưu nhất**

- **Nhận diện phân lớp ô nhiễm cực đoan (Lớp 3):** Mặc dù Random Forest (Tuned) có điểm số F1-Macro tổng thể cao hơn, nhưng thuật toán này bỏ sót hoàn toàn các ngày thuộc Lớp 3 (Mức Nguy hại). Trong bài toán sức khỏe cộng đồng, việc bỏ sót ngày ô nhiễm cực đoan mang lại rủi ro lớn hơn việc dự báo nhầm. LightGBM với cơ chế phát triển cây theo cấu trúc lá (Leaf-wise) là mô hình duy nhất nhận diện được Lớp 3.

- **Sự ổn định trên các phân lớp khác:** Mô hình duy trì độ chính xác và độ bao phủ cao ở Lớp 1, đồng thời giữ được mức cân bằng đạt yêu cầu ở Lớp 2.

#### **Đánh giá hạn chế của dự án**

- **Sự mất cân bằng dữ liệu:** Số lượng mẫu Lớp 3 trong tập kiểm thử quá ít (5 dòng dữ liệu) khiến chỉ số F1-Macro chung của LightGBM bị kéo thấp xuống do mô hình chưa có đủ thông tin để học các đặc trưng cực đoan.

- **Thiếu hụt đặc trưng khí tượng:** Ranh giới phân loại bị nhiễu do tập dữ liệu chỉ chứa nồng độ chất ô nhiễm quá khứ. Việc thiếu các biến động lực học như tốc độ gió, hướng gió, độ ẩm hoặc lượng mưa khiến cấu trúc cây quyết định gặp khó khăn khi thời tiết biến động đột ngột.

#### **Hướng phát triển tiếp theo**

- **Xử lý dữ liệu lệch:** Áp dụng kỹ thuật tái lấy mẫu SMOTE để tự động sinh thêm mẫu giả lập cho Lớp 3, tăng không gian huấn luyện cho LightGBM.

- **Làm giàu đặc trưng:** Tích hợp các yếu tố thời tiết thực tế thực hiện qua OpenWeatherMap API (gió, mưa, độ ẩm) nhằm làm rõ ranh giới phân loại giữa các cấp độ AQI.

- **Triển khai ứng dụng:** Đóng gói mô hình LightGBM đã tối ưu hóa thành công cụ dự báo và xây dựng giao diện bảng điều khiển trực quan bằng Streamlit để hỗ trợ theo dõi thực tế.

#### **Lưu best classifier**

In [ ]:
# Lưu best model chính thức
best_model = best_lgbm

models_dir = os.path.join(ROOT, "models")
os.makedirs("models", exist_ok=True)
save_path = os.path.join(models_dir, "best_classifier.pkl")
joblib.dump(best_model, save_path)

print("Đã lưu best_classifier.pkl")

## **04. Model Classification**

#### **Mục tiêu:**

- Tính toán SHAP values cho Best Model đã chọn ở phần Regression.
- Vẽ Summary Plot để xác định feature nào quan trọng nhất đối với toàn bộ tập dữ liệu.
- Vẽ Beeswarm Plot để hiểu chiều hướng tác động (giá trị feature cao/thấp đẩy AQI lên hay xuống).
- Vẽ Waterfall Plot để giải thích chi tiết một dự đoán cụ thể.
- Vẽ Dependence Plot để xem mối quan hệ tương tác giữa 2 features.
- So sánh kết quả SHAP với Feature Importance đã tính ở phần Regression để đối chiếu.

#### **Cấu trúc:**
1. SHAP Values
2. Summary Plot - Tổng quan tầm quan trọng tổng thể
3. Beeswarm - Chiều hướng tác động
4. Waterfall Plot - Giải thích 1 dự đoán cụ thể
5. Dependence Plot - Mối quan hệ giữa 2 Features
6. So sánh SHAP với Feature Importance (Split/Gain)
7. Kết luận SHAP

In [ ]:
models_ID = "1N7bZA7lbf-DW2k-urlCpw967sUT3aKu-"
models_url = f"https://drive.google.com/drive/folders/{models_ID}"

gdown.download_folder(models_url, output="models", quiet=False)

best_model = joblib.load("models/best_regressor.pkl")
feature_cols = joblib.load("models/feature_cols.pkl")

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(
    f"Khoảng thời gian Test: {X_test.index.min().date()} → {X_test.index.max().date()}"
)

## **05. SHAP Explainability**



**SHAP (SHapley Additive exPlanations)** là phương pháp giải thích mô hình dựa trên lý thuyết trò chơi hợp tác (cooperative game theory) - cụ thể là giá trị Shapley. **Ý tưởng cốt lõi:** Mỗi feature được xem như một "người chơi" đóng góp vào kết quả dự đoán, và SHAP value đo lường chính xác mức đóng góp đó.

**Công thức trực quan:**

$$\text{Dự đoán} = \text{Base value} + \sum_{i=1}^{n} \text{SHAP}_i$$

Trong đó:
- **Base value**: Giá trị dự đoán trung bình nếu không biết bất kỳ thông tin gì (tương đương AQI trung bình của tập Train).
- **SHAP$_i$**: Mức đóng góp của feature thứ $i$ - dương nghĩa là đẩy dự đoán lên cao hơn Base value, âm nghĩa là kéo xuống thấp hơn Base value.

**Tại sao dùng SHAP thay vì chỉ dùng Feature Importance (Split/Gain)?**
- Feature Importance chỉ cho biết feature nào **quan trọng**, không cho biết **quan trọng theo chiều nào** (tăng hay giảm AQI).
- SHAP giải thích được **từng dự đoán cụ thể**, không chỉ tổng quan toàn bộ model.
- SHAP có nền tảng lý thuyết toán học chặt chẽ (tính nhất quán - consistency), không bị thiên vị bởi loại feature (liên tục vs nhị phân) như Split Importance.

### **5.1. SHAP Values**

In [ ]:
# TreeExplainer - tối ưu cho các mô hình dựa trên cây (XGBoost, LightGBM, Random Forest)
# Nếu Best Model là Ridge (Linear), cần dùng shap.LinearExplainer thay thế

model_type = type(best_model).__name__

if "Ridge" in model_type or "Linear" in model_type:
    print(f"Best Model là {model_type} (Linear) - dùng shap.LinearExplainer")
    masker = shap.maskers.Independent(X_train, max_samples=X_train.shape[0])
    explainer = shap.LinearExplainer(best_model, masker)
else:
    print(f"Best Model là {model_type} (Tree-based) - dùng shap.TreeExplainer")
    explainer = shap.TreeExplainer(best_model)

# Tính SHAP values trên tập Test
shap_values = explainer(X_test)

print(f"   Shape: {shap_values.values.shape}")
print(
    f"   Base value: {shap_values.base_values[0] if hasattr(shap_values.base_values, '__len__') else shap_values.base_values:.2f}"
)

#### **Nhận xét chung:**

- Base value xấp xỉ AQI trung bình của tập Train (~78.75) - đây là điểm cơ sở trước khi xét đến đặc điểm riêng của từng ngày.
- Mỗi dòng trong `shap_values.values` tương ứng với 1 ngày trong tập Test, mỗi cột là mức đóng góp của 1 feature cho ngày đó.

### **5.2. Summary Plot - Tổng quan tầm quan trọng tổng thể**

Summary Plot (dạng bar) xếp hạng các features theo **Mean |SHAP Value|** - trung bình giá trị tuyệt đối của SHAP trên toàn bộ tập Test. Feature có thanh dài nhất là feature ảnh hưởng nhiều nhất đến dự đoán, dù đó là ảnh hưởng tăng hay giảm AQI.

In [ ]:
best_model_name = "Ridge Regression"

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=20, show=False)
plt.title(
    f"SHAP Summary Plot - {best_model_name}\nTop 20 Features Quan Trọng Nhất",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "SHAP_summary_plot")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Nhận xét Summary Plot:**

- **Feature đứng đầu:** `pm2_5` với mức độ ảnh hưởng trung bình (mean |SHAP value|) hơn 17.5. Kết quả này hoàn toàn khớp với kỳ vọng của nhóm. Giá trị SHAP của pm2_5 vượt trội hoàn toàn so với các biến còn lại (gấp hơn 3 lần so với biến đứng thứ hai). Điều này cho thấy nồng độ PM2.5 hiện tại có sức nặng quyết định lớn nhất đến kết quả dự đoán của mô hình Ridge Regression.

- **So sánh top 5 features của SHAP với Feature Importance của các mô hình hồi quy:**

| | SHAP | Ridge | RF | DT | LGBM | XGB |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | pm2_5 | pm2_5 | pm2_5 | pm2_5 | pm2_5 | pm2_5 |
| 2 | diff_ 1 |  diff_ 1 | pm10 | rolling_std_30 | diff_ 1 | diff_ 1 |
| 3 | nitrogen_dioxide | ozone | nitrogen_dioxide | carbon_monoxide | diff_ 7 | diff_ 7 |
| 4 | ozone | nitrogen_dioxide | carbon_monoxide | pm10 | carbon_monoxide | carbon_monoxide |
| 5 | t-1 | t-1 | t-1 | sulphur_dioxide | ozone | ozone |

> **Kết luận:** Nếu cần chọn ra một tập Core Features để làm gọn mô hình mà không làm giảm hiệu suất, nhóm sẽ chọn:
> - Biến cốt lõi: pm2_5 (Bắt buộc).
> - Biến động lượng (Trend/Lag): diff_1 (xuất hiện ở 4/6 mô hình), diff_7 (quan trọng với Tree mạnh), và t-1.
> - Biến khí tượng / hóa học: ozone, carbon_monoxide, và nitrogen_dioxide. Các mô hình khác nhau sẽ tự biết cách tận dụng một trong số các loại khí này để tối ưu hóa kết quả.

### **5.3. Beeswarm - Chiều hướng tác động**

Khác với Summary Plot dạng bar (chỉ cho biết "quan trọng bao nhiêu"), Beeswarm Plot cho biết thêm **chiều hướng tác động**:
- Mỗi chấm là 1 ngày trong tập Test.
- Vị trí trên trục X: Giá trị SHAP của ngày đó (âm = kéo AQI xuống, dương = đẩy AQI lên).
- Màu sắc: Giá trị thực tế của feature đó (đỏ = giá trị cao, xanh = giá trị thấp).

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, max_display=20, show=False)
plt.title(
    f"SHAP Beeswarm Plot - {best_model_name}\nĐỏ = Giá Trị Feature Cao | Xanh = Giá Trị Feature Thấp",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "SHAP_beeswarm_plot")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Nhận xét Beeswarm Plot:**

- **Đối với feature t-1 (và feature đứng đầu pm2_5):** Nồng độ pm2_5 hiện tại mới là yếu tố trực tiếp đẩy dự đoán AQI lên cao (chấm đỏ bên phải). Trong khi đó, trong bối cảnh đã biết pm2_5, nếu t-1 cao thì mô hình sẽ có xu hướng hãm kết quả dự đoán lại một chút.

- **Đối với feature is_dry_season:** Ở hàng is_dry_season, các chấm màu đỏ (giá trị = 1, tức là mùa khô) lại tụ tập ở bên trái (SHAP value âm). Các chấm xanh (mùa mưa) nằm ở mức 0 hoặc hơi dương nhẹ. Điều này có vẻ trái ngược với những phát hiện từ EDA (thường mùa khô sẽ có AQI cao hơn do không có mưa rửa trôi bụi). Tuy nhiên, đây là sự khác biệt giữa tác động biên (EDA) và tác động cục bộ (Mô hình). Sau khi mô hình đã biết rõ các thông số nồng độ bụi (pm2_5, pm10) và khí độc (Ozone, NO2...) của ngày hôm đó, thì việc dán nhãn "ngày này thuộc mùa khô" không làm tăng thêm mức độ ô nhiễm nữa, mà thậm chí đóng vai trò như một hệ số bù trừ làm giảm nhẹ kết quả xuống.

> **Kết luận:** Biểu đồ SHAP của mô hình Ridge cho thấy rõ đặc tính tác động cục bộ. Nó không thể hiện mối quan hệ tuyệt đối như EDA, mà thể hiện: "Khi giữ nguyên tất cả các thông số khác (đặc biệt là nồng độ các chất ô nhiễm), biến này sẽ tác động thêm vào dự đoán như thế nào?".

### **5.4. Waterfall Plot - Giải thích 1 dự đoán cụ thể**

Waterfall Plot trả lời câu hỏi cụ thể: **"Tại sao mô hình dự đoán AQI = X cho ngày này?"** - bằng cách hiển thị từng feature đóng góp bao nhiêu, cộng dồn từ Base value đến giá trị dự đoán cuối cùng.

Ở đây nhóm chọn ngày có **AQI dự đoán cao nhất** trong tập Test để minh họa.

In [ ]:
# Tìm ngày có AQI dự đoán cao nhất trong tập Test
y_pred_test = np.clip(best_model.predict(X_test), 0, 500)
idx_max = int(np.argmax(y_pred_test))
date_max = X_test.index[idx_max]

actual_aqi = float(np.ravel(y_test.iloc[idx_max])[0])
predicted_aqi = float(np.ravel(y_pred_test[idx_max])[0])

print(
    f"Ngày được chọn để giải thích: {date_max.strftime('%d/%m/%Y') if hasattr(date_max, 'strftime') else date_max}"
)
print(f"AQI thực tế   : {actual_aqi:.1f}")
print(f"AQI dự đoán   : {predicted_aqi:.1f}")

In [ ]:
plt.figure(figsize=(10, 7))
shap.waterfall_plot(shap_values[idx_max], max_display=14, show=False)
date_label = (
    date_max.strftime("%d/%m/%Y") if hasattr(date_max, "strftime") else str(date_max)
)
plt.title(
    f"SHAP Waterfall Plot - Ngày {date_label}\nAQI Dự Đoán = {y_pred_test[idx_max]:.0f}",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "SHAP_waterfall_plot")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Phân tích Waterfall Plot**

Ngày [19/06/2025]: AQI dự đoán = [155.506]
- Bắt đầu từ Base value = [78.745] (AQI trung bình tập Train)
- `pm2_5` = [3.41] → [+84.59] (Nồng độ bụi mịn trong ngày rất cao, đây là yếu tố chính đẩy dự đoán AQI vọt lên mức ô nhiễm).
- `diff_1` = [3.05] → [-18.65] (Biến động lượng đóng vai trò hãm mạnh nhất để dự đoán không bị quá lố).
- `nitrogen_dioxide` = [1.603] → [+8.61] (Nồng độ khí NO2 cao tiếp tục đẩy mức độ ô nhiễm lên).
- `diff_7` = [1.797] → [-7.61] (Biến động lượng chu kỳ 7 ngày giúp điều chỉnh giảm).
- `t-1` = [1.028] → [-4.38] (Mô hình sử dụng AQI hôm qua như một hệ số bù trừ nhẹ, kéo giảm dự đoán xuống).
- Các features như is_dry_season, rolling_mean_7... có mức độ ảnh hưởng rất thấp trong ngày này, được gộp chung vào nhóm "27 other features" với tổng tác động là -0.1.
- Tổng cộng → Dự đoán cuối cùng = [155.506].

> Đây là cách mô hình hoạt động.

### **5.5. Dependence Plot - Mối quan hệ giữa 2 Features**

Dependence Plot cho thấy mối quan hệ giữa **giá trị của 1 feature** và **SHAP value tương ứng** của nó, đồng thời tô màu theo 1 feature thứ 2 để phát hiện tương tác (interaction) giữa 2 features.

Ở đây nhóm xem xét feature quan trọng nhất và tương tác của nó với `is_dry_season`.

In [ ]:
# Xác định feature quan trọng nhất để vẽ Dependence Plot
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top_feature = X_test.columns[np.argmax(mean_abs_shap)]
print(f"Feature quan trọng nhất: {top_feature}")

# Feature thứ 2 để tô màu tương tác - ưu tiên is_dry_season nếu có
interaction_feature = "is_dry_season" if "is_dry_season" in X_test.columns else None

In [ ]:
plt.figure(figsize=(10, 6))
shap.dependence_plot(
    top_feature,
    shap_values.values,
    X_test,
    interaction_index=interaction_feature,
    show=False,
)
plt.title(
    f"SHAP Dependence Plot x {top_feature}"
    + (f" x {interaction_feature}" if interaction_feature else ""),
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "SHAP_dependence_plot")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Nhận xét Dependence Plot:**

- **Xu hướng tổng thể:** Có mối quan hệ đồng biến hoàn hảo (tuyến tính). Khi giá trị thực tế của pm2_5 tăng lên, mức độ đóng góp (SHAP value) của nó vào kết quả dự đoán cũng tăng theo tỷ lệ thuận. Các điểm dữ liệu xếp thành một đường thẳng tắp, minh chứng rõ nét cho bản chất phương trình bậc nhất của mô hình Ridge Regression.
- **Tương tác với is_dry_season:** Hoàn toàn không có sự tương tác giữa 2 biến này trong mô hình hiện tại. Tại bất kỳ một mức nồng độ pm2_5 nào (cùng một giá trị trục X), các điểm màu đỏ (mùa khô) và màu xanh (mùa mưa) đều nằm đè lên cùng một đường thẳng duy nhất, không hề có sự phân tách hay chênh lệch độ cao. Lý do là vì Ridge Regression là mô hình cộng tính độc lập. Nếu không tự tạo ra một biến tương tác nhân (ví dụ: pm2_5 * is_dry_season) ở bước tiền xử lý, mô hình sẽ mặc định tác động của bụi là độc lập và không bị thay đổi bởi yếu tố thời tiết.

### **5.6. So sánh SHAP với Feature Importance (Split/Gain)**


Ở phần huấn luyện mô hình đã tính Feature Importance theo phương pháp Split (đếm số lần feature được dùng để chia nhánh) - phương pháp này có thể thiên vị features có nhiều giá trị liên tục. Ở đây nhóm so sánh trực tiếp với SHAP (phương pháp công bằng hơn về mặt lý thuyết) để xem có sự khác biệt lớn không.

In [ ]:
# 1. Xếp hạng theo SHAP
fi_shap = pd.Series(mean_abs_shap, index=X_test.columns).sort_values(ascending=False)
comparison_data = {"Feature": fi_shap.index, "SHAP_rank": range(1, len(fi_shap) + 1)}

# 2. Xếp hạng theo Model Gốc
fi_model = None

# Nếu là mô hình Tree-based
if hasattr(best_model, "feature_importances_"):
    fi_model = pd.Series(best_model.feature_importances_, index=X_test.columns)
# Nếu là mô hình Linear (như Ridge)
elif hasattr(best_model, "coef_"):
    # Lấy trị tuyệt đối của trọng số vì ta chỉ quan tâm độ lớn tác động
    fi_model = pd.Series(np.abs(best_model.coef_), index=X_test.columns)

# 3. Ghép kết quả và tính chênh lệch
if fi_model is not None:
    fi_model = fi_model.sort_values(ascending=False)
    model_rank_map = {feat: i + 1 for i, feat in enumerate(fi_model.index)}
    comparison_data["Model_rank"] = [model_rank_map.get(f, None) for f in fi_shap.index]

df_compare = pd.DataFrame(comparison_data).head(15)

# Tính chênh lệch thứ hạng
if "Model_rank" in df_compare.columns:
    df_compare["Rank_diff"] = (df_compare["Model_rank"] - df_compare["SHAP_rank"]).abs()

print("So sánh Top 15 Features: SHAP vs Feature Importance")
display(df_compare)

### **5.7. Kết luận SHAP**

| Bước | Kết quả |
|------|---------|
| Tính SHAP values | TreeExplainer/LinearExplainer trên tập Test |
| Summary Plot | Xác định feature quan trọng nhất |
| Beeswarm Plot | Xác định chiều hướng tác động |
| Waterfall Plot | Giải thích 1 dự đoán cụ thể (ngày AQI cao nhất) |
| Dependence Plot | Phát hiện tương tác giữa 2 features |
| So sánh với Split Importance | Xác nhận tính nhất quán giữa 2 phương pháp |

#### **Kết luận tổng thể:**
- **Top 3 features quan trọng nhất theo SHAP:**
1.  pm2_5: Yếu tố cốt lõi mang tính quyết định tuyệt đối đến dự đoán AQI.
2.  diff_1: Biến động lượng đóng vai trò hãm và điều chỉnh sai số.
3.  nitrogen_dioxide (NO2): Loại khí thải độc hại có sức nặng lớn thứ hai sau bụi mịn.

- **Sự đồng thuận đa chiều (SHAP vs. EDA vs. Model):**
1. Khớp với Model Coefficients: Có sự nhất quán gần như tuyệt đối (chênh lệch hạng = 0) ở 7 biến quan trọng nhất giữa phương pháp giải thích SHAP và trọng số toán học nội tại của mô hình Ridge. Điều này chứng minh mô hình học được các quy luật rất vững chắc, không bị nhiễu.

2. Khớp với phần khám phá dữ liệu (EDA): Phân tích đa biến của SHAP đã giải thích sâu hơn những gì EDA nhìn thấy. Trong EDA (đơn biến), ta thấy t-1 (AQI hôm qua) và mùa khô tỷ lệ thuận với mức độ ô nhiễm. Tuy nhiên, SHAP tiết lộ rằng trong mô hình dự đoán, một khi đã biết chính xác nồng độ pm2_5 của ngày hôm nay, mô hình sẽ dùng t-1 và các biến chu kỳ như hệ số bù trừ để tránh hiện tượng cộng dồn thông tin, giúp dự đoán không bị vọt lên lố đà. Sự tương tác mùa vụ (is_dry_season) cũng bị lu mờ hoàn toàn trước sức ảnh hưởng trực tiếp của nồng độ bụi.

- **Ý nghĩa thực tiễn (Sự minh bạch của mô hình):**
1. Phá vỡ Hộp đen (Black-box): Biểu đồ Waterfall có thể giải thích minh bạch đến từng điểm dự đoán cho bất kỳ ngày cụ thể nào: Bắt đầu từ mức trung bình là bao nhiêu, bị đẩy lên do đâu (như bụi mịn, NO2), và được kéo xuống nhờ yếu tố nào (như biến động lượng).

2. Tính logic và độ tin cậy: Biểu đồ Dependence Plot vẽ ra một đường thẳng hoàn hảo cho pm2_5, xác nhận Ridge Regression đang hoạt động cực kỳ ổn định, đúng với bản chất tuyến tính của nó. Hoàn toàn có thể tin tưởng rằng mô hình đưa ra dự đoán dựa trên các cơ sở khoa học về môi trường một cách hợp lý, chứ không phải học vẹt từ các điểm dữ liệu nhiễu.